# RSNA Intracranial Hemorrhage Detection

### Link: https://www.kaggle.com/competitions/rsna-intracranial-hemorrhage-detection

### 🔬 Radiological Preprocessing: Mapping DICOM Windows (HU) to RGB Channels
Unlike standard computer vision algorithms, medical pixel fidelity is preserved by mapping the high dynamic range CT data into a 3-channel RGB format utilizing medically validated Hounsfield Unit (HU) windows: **Brain Window (Red), Subdural Window (Green), and Bone Window (Blue)**. This radiological intervention maximizes the contrast of hemorrhage textures for the Convolutional Neural Network (CNN).

In [ ]:
import pydicom
import glob
import numpy as np
import matplotlib.pyplot as plt

arama_yolu = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/*.dcm"
dosyalar = glob.glob(arama_yolu)

if len(dosyalar) > 0:
    hedef_dosya = dosyalar[0]
    print("Kilit kırıldı, dosya negatif HU taramasıyla işleniyor:", hedef_dosya)
    
    # --- Tıbbi Pencereleme Fonksiyonlarımız ---
    def get_first(x):
        if type(x) == pydicom.multival.MultiValue: return int(x[0])
        return int(x)

    def window_image(img, center, width, intercept, slope):
        img = (img * slope + intercept)
        img_min = center - (width // 2)
        img_max = center + (width // 2)
        img[img < img_min] = img_min
        img[img > img_max] = img_max
        return (img - img_min) / (img_max - img_min)

    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]
    _, _, inter, slope = [get_first(x) for x in fields]
    
    # İŞTE HAYAT KURTARAN TIP MÜHENDİSLİĞİ DOKUNUŞU:
    # "Negatif Doku yoğunluklarına (Hava = -1024) izin ver"
    img = dataset.pixel_array.astype(float)
    
    red = window_image(img.copy(), 40, 80, inter, slope)       # Brain / Beyin Dokusu (Kırmızı)
    green = window_image(img.copy(), 80, 200, inter, slope)    # Subdural Bölge (Yeşil)
    blue = window_image(img.copy(), 600, 2800, inter, slope)   # Bone / Kafatası (Mavi)
    
    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
    
    # --- Ekrana Çizdirme ---
    plt.figure(figsize=(10,10))
    plt.imshow(rgb)
    plt.title("RSNA Brain - Subdural - Bone CT Window")
    plt.axis('off')
    plt.show()


In [ ]:
import pandas as pd

# Cevap anahtarının tam klasör yolu
csv_yolu = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train.csv"

print("Cevap anahtarı okunuyor... (Bu tablo biraz büyük, saniyeler sürebilir)")
tanilar_tablosu = pd.read_csv(csv_yolu)

print("Klinik Tablo Yüklendi! Toplam Kayıt Sayısı:", len(tanilar_tablosu))

# Tablonun en üstündeki ilk 10 radyoloji raporunu görelim:
tanilar_tablosu.head(10)


In [ ]:
print("Veriler tıbbi hasta dosyası matrisine (Pivot) dönüştürülüyor. Bu işlem yaklaşık 10-15 saniye sürebilir...")

# 1. Aynı dosyadan mükerrer girmeler yapılmışsa onları silelim (Kaggle verisinde bu hata çok olur)
tanilar_tablosu = tanilar_tablosu.drop_duplicates(subset=['ID']).copy()

# 2. 'ID' sütunundaki "_" işaretinden bölerek "Hasta_ID" ve "Kanama_Tipi" olarak ikiye ayıralım
tanilar_tablosu[['Slice_ID', 'Hemorrhage_Type']] = tanilar_tablosu['ID'].str.rsplit('_', n=1, expand=True)

# 3. Pivot Yapalım: Herkesin tek bir satırı olsun, Kanama tipleri yanına sütun olarak dizilsin
hasta_matrisi = tanilar_tablosu.pivot(index='Slice_ID', columns='Hemorrhage_Type', values='Label')

# Kafa karışıklığını önlemek için sadece kanama sütunlarını öne alalım
hasta_matrisi = hasta_matrisi[['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']]

print("İşlem Başarılı! İşte Yapay Zekanın Aşık Olacağı O Harika Tıbbi Format:")
hasta_matrisi.head(10)


In [ ]:
# matriste herhangi bir kanaması (any == 1) olan hastaları filtreleyelim
kanamali_hastalar = hasta_matrisi[hasta_matrisi['any'] == 1]

# Kanaması olan İLK hastanın numarasını alalım (Örn: ID_xxxxxx)
kanamali_hasta_id = kanamali_hastalar.index[0]

print("🚨 DİKKAT: Kanamalı Hasta Bulundu!")
print("Hasta Numarası:", kanamali_hasta_id)
print("\nUzmanların Koyduğu Teşhis Tablosu:")
print(kanamali_hastalar.loc[kanamali_hasta_id])

# Şimdi adresini bulup resmini de ekrana basalım
hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{kanamali_hasta_id}.dcm"

# Önceki kodumuzda kullandığımız pencereleri aynen çalıştırıyoruz
dataset = pydicom.dcmread(hedef_dosya)
fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
          dataset[('0028','1052')].value, dataset[('0028','1053')].value]
_, _, inter, slope = [get_first(x) for x in fields]

img = dataset.pixel_array.astype(float)
red = window_image(img.copy(), 40, 80, inter, slope)       
green = window_image(img.copy(), 80, 200, inter, slope)    
blue = window_image(img.copy(), 600, 2800, inter, slope)   

rgb = np.zeros((img.shape[0], img.shape[1], 3))
rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

plt.figure(figsize=(10,10))
plt.imshow(rgb)
plt.title(f"Pozitif Kanama Tespiti: {kanamali_hasta_id}")
plt.axis('off')
plt.show()


In [ ]:
import math
import cv2
import tensorflow as tf
from tensorflow.keras.utils import Sequence

print("Taşıyıcı Bant Sistemi (Data Generator) İnşa Ediliyor...")

# Bandın Ana Mekanizması
class RSNADataset(Sequence):
    def __init__(self, hasta_listesi, etiket_matrisi, batch_size=32, img_size=(256, 256)):
        self.hasta_listesi = hasta_listesi
        self.etiket_matrisi = etiket_matrisi
        self.batch_size = batch_size
        self.img_size = img_size
        self.dcm_dir = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/"
        
    def __len__(self):
        # Tüm hastaların kaç koliye sığacağını hesaplar
        return math.ceil(len(self.hasta_listesi) / self.batch_size)
    
    def __getitem__(self, idx):
        # Yapay zeka ne zaman "Sıradaki koli gelsin!" dese bu kısım çalışır
        batch_ids = self.hasta_listesi[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        # X: Görüntülerin yerleştirileceği kutu (Koli boyutu, En, Boy, 3 Renk kanalı)
        X = np.zeros((len(batch_ids), self.img_size[0], self.img_size[1], 3)) 
        # Y: Tıbbi teşhislerin listesi (Koli boyutu, 6 Farklı hastalık sütunu)
        Y = np.zeros((len(batch_ids), 6))
        
        for i, p_id in enumerate(batch_ids):
            dcm_path = self.dcm_dir + str(p_id) + ".dcm"
            try:
                # Tomografiyi okuyoruz
                dataset = pydicom.dcmread(dcm_path)
                fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                          dataset[('0028','1052')].value, dataset[('0028','1053')].value]
                _, _, inter, slope = [get_first(x) for x in fields]
                
                img = dataset.pixel_array.astype(float)
                
                # 3 Tıbbi Pencereden geçiriyoruz (Sizin uzmanlığınız olan adım)
                red = window_image(img.copy(), 40, 80, inter, slope)
                green = window_image(img.copy(), 80, 200, inter, slope)
                blue = window_image(img.copy(), 600, 2800, inter, slope)
                
                rgb = np.zeros((img.shape[0], img.shape[1], 3))
                rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
                
                # CT Görüntüleri normalde çok büyüktür (512x512). Yapay zeka hızlansın diye 256x256 yapıyoruz.
                rgb_resized = cv2.resize(rgb, self.img_size)
                
                X[i,] = rgb_resized
                Y[i,] = self.etiket_matrisi.loc[p_id].values
                
            except Exception as e:
                # Eğer Kaggle depolarında dosyası bozuk bir hasta varsa sistemi çökertmemek için görmezden gelir
                pass 
                
        return X, Y

print("✅ Fabrika İnşa Edildi!")

# ----------------- SİSTEM TESTİ -----------------
# Tablomuzdaki tüm hastaların numara listesini alalım
tam_hasta_listesi = hasta_matrisi.index.tolist()

# Bandı test edelim: Bir koliye sadece 8 hasta sığsın
test_bandi = RSNADataset(tam_hasta_listesi, hasta_matrisi, batch_size=8)

# Düğmeye basıp banttan sıradaki İLK koliyi alalım:
ornek_resimler, ornek_teşhisler = test_bandi[0]

print("\n--- İlk Koli Teslim Alındı ---")
print("Röntgen Paketinin Çapı:", ornek_resimler.shape)
print("Teşhis Paketinin Çapı:", ornek_teşhisler.shape)


### 🧱 Stage 1 Training: Addressing Class Imbalance & The 'Amnesia' Trap
The overwhelming prevalence of the "Healthy" class in the raw dataset often causes the model to take a shortcut, memorizing cranial boundaries instead of learning pathology. To prevent this "Amnesia Trap", the clinical dataset was artificially Balanced to a 50/50 ratio. Furthermore, the pre-trained EfficientNet base layers were **Frozen (Frozen Base)**. This forces the model to synthesize generic shape recognition into a specialized hemorrhage decision-lobe rather than simply overfitting to the skull contours.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

print("Uzman Zeka (EfficientNet) Laboratuvara Transfer Ediliyor...")

# 1. Eski tecrübeli beyni (EfficientNetB0) alıyoruz.
# include_top=False: Eski karar mekanizmasını (köpek/kedi/araba tanıma kısmı) kesip atar.
# weights='imagenet': Şekilleri ve kenarları önceden öğrenmiş vizyon tecrübesini korur.
temel_beyin = EfficientNetB0(input_shape=(256, 256, 3), include_top=False, weights='imagenet')

# 2. Beynin üzerine kendi karar mekanizmamızı (Hemorrhage) ekliyoruz
x = temel_beyin.output
x = GlobalAveragePooling2D()(x) # Trilyonlarca piksel bilgisini hücre seviyesinde özetler ve sıkıştırır
x = Dropout(0.3)(x)             # Tıpta "Aşırı ezberlemeyi" (Overfitting) önlemek çok önemlidir! Sisteme ufacık bir unutkanlık eşiği ekliyoruz.

# 3. Nihai Karar Lobu: 6 farklı hastalık (kanama) tipi için bağımsız 6 adet çıkış vanası!
# Sigmoid kullanıyoruz ki: Hem Subdural 1, hem Epidural 1 olabilsin. Birbirini engellemesinler.
kesin_tanilar = Dense(6, activation='sigmoid')(x)

# Vücutla (Temel Beyin) Yeni Kafayı (Nihai Lob) cerrahi olarak birleştiriyoruz
bizim_model = Model(inputs=temel_beyin.input, outputs=kesin_tanilar)

# 4. Asistanın çalışmaya başlama / öğrenme talimatları
# tf.keras.optimizers.Adam: Hangi hızla öğreneceğini belirler (çok yavaş olursa vakit biter, çok hızlı olursa dersleri anlamaz)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005) 

# Sistemi derleyelim
bizim_model.compile(optimizer=optimizer, 
                    loss='binary_crossentropy', # Çok seçmeli ve birden fazla cevabı olabilecek sınavlar (Multi-label) için en iyi puanlama yöntemi
                    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)]) # Doğruluk yüzdesi ve AUC skoru çetelesi

print("\n✅ Ve Asistanımızın Yeni Beyni Cerrahi Olarak Hazırlandı!")
print("İşte beynin katmanlarının röntgeni (Özeti):")
bizim_model.summary()

In [ ]:
print("Eğitim Süreci (Asistan Asistanlığı) Başlıyor...")

# Taşıyıcı bantımızı standart fabrika ayarlarına (Koli başı 32 Hasta) getirelim
orijinal_egitim_bandi = RSNADataset(tam_hasta_listesi, hasta_matrisi, batch_size=32)

# Asistanın derse oturduğu an! 
# Epochs: Kitabı baştan sona kaç kere okuyacağı (1 yeterli)
# steps_per_epoch: Kitabın tamamını değil de, şimdilik sadece ilk 50 sayfayı (koliyi) okuyup bırakması için.
history = bizim_model.fit(
    orijinal_egitim_bandi, 
    epochs=1,
    steps_per_epoch=50,
    verbose=1
)

print("\n🎯 Harika! Asistan 1600 hastayı (50 Koli) dakikalar içinde inceledi ve eğitim aşamasını kavradı.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Daha önce bulduğumuz o kanamalı hastayı (ID_27a354d42) tekrar viziteye çağırıyoruz
secili_hasta_id = kanamali_hasta_id  

# Masadaki Gerçek Tıbbi Raporumuz (Cevap Anahtarımız)
gercek_tanilar = hasta_matrisi.loc[secili_hasta_id]

# Hastanın Tomografisini Arşivden Alalım
hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + secili_hasta_id + ".dcm"

# Hızlıca Pencereleri Çizelim (Asistanın gözlüğünü takalım)
dataset = pydicom.dcmread(hedef_dosya)
fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
          dataset[('0028','1052')].value, dataset[('0028','1053')].value]
_, _, inter, slope = [get_first(x) for x in fields]

img = dataset.pixel_array.astype(float)
red = window_image(img.copy(), 40, 80, inter, slope)
green = window_image(img.copy(), 80, 200, inter, slope)
blue = window_image(img.copy(), 600, 2800, inter, slope)

rgb = np.zeros((img.shape[0], img.shape[1], 3))
rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

# Yapay Zeka için yeniden boyutlandırma
rgb_resized = cv2.resize(rgb, (256, 256))

# Yapay zekaya bu resmi sanki 1 kişilik bir yığınmış (batch) gibi klasöre koyup gönderelim
yapay_zeka_verisi = np.expand_dims(rgb_resized, axis=0)

# ================================
# ASİSTANIN FİKRİNİ ALIYORUZ!
tahminler = bizim_model.predict(yapay_zeka_verisi)
# ================================

kanama_tipleri = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

print(f"\n=============================================")
print(f"   👨‍⚕️ UZMAN DOKTOR VS 🤖 YAPAY ZEKA   ")
print(f"   HASTA: {secili_hasta_id}")
print(f"=============================================\n")

for i, tip in enumerate(kanama_tiplerizzz):
    # Eğer cevap anahtarı 1 ise VAR, 0 ise YOK yazalım
    gercek_sonuc = "VAR" if gercek_tanilar[tip] == 1 else "Yok "
    
    # Asistanın cevabını (0.00 ile 1.00 arası değişen sayıyı) Yüzdeye (%) çevirelim
    asistan_tahmini = tahminler[0][i] * 100 
    
    # Karşılaştırmalı Ekrana Yazdıralım
    print(f"🩸 {tip.ljust(16).upper()}   | 👨‍⚕️ UZMAN: {gercek_sonuc.ljust(4)} | 🤖 ASİSTAN: % {asistan_tahmini:.1f} Şüphe")

# Ve Görsel Olarak Hastayı da Gösterelim
plt.figure(figsize=(7,7))
plt.imshow(rgb)
plt.title(f"Clinical Audit: {secili_hasta_id}")
plt.axis('off')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

print("Bilimsel Kanıt (Learning Curve) Aşaması Başlıyor...")

# Asistanın okuyacağı kitap sayısını artırıyoruz. 
# 5 Tur (Epoch) boyunca hastaları tekrar tekrar inceleyecek ve her seferinde kendini düzeltecek.
GELISMIS_EGITIM_TURU = 5
GELISMIS_TEST_ADIMI = 50 # Her turda 50 koli tomografi okuyacak

# Asistan ders programına (Fit) geri dönüyor
history = bizim_model.fit(
    orijinal_egitim_bandi, 
    epochs=GELISMIS_EGITIM_TURU,
    steps_per_epoch=GELISMIS_TEST_ADIMI,
    verbose=1
)

# -----------------------------------------------------
# EĞİTİM BİTTİ -> ŞİMDİ BUNU KLİNİK GRAFİĞE DÖKÜYORUZ
# -----------------------------------------------------
plt.figure(figsize=(14, 5))

# 1. Grafik: Asistanın Hata Payı (Loss) - Zamanla düşmesi beklenir!
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Hata Skoru', color='red', linewidth=2, marker='o')
plt.title('Asistanın Yanılma Payı (Loss Göstergesi)')
plt.xlabel('Eğitim Turu (Epochs)')
plt.ylabel('Hata Miktarı')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# 2. Grafik: Asistanın Teşhis Başarısı (Accuracy ve AUC) - Zamanla yükselmesi beklenir!
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Genel Doğruluk', color='green', linewidth=2, marker='s')
plt.plot(history.history['auc'], label='Klinik Hassasiyet (AUC)', color='blue', linewidth=2, marker='^')
plt.title('Asistanın Doğru Teşhis Kapasitesi')
plt.xlabel('Eğitim Turu (Epochs)')
plt.ylabel('Başarı Yüzdesi (0.0 - 1.0)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

print("\n✅ Mükemmel! Asistanın zaman geçtikçe nasıl uzmanlaştığını bilimsel grafiklerle ispatladınız.")


In [ ]:
import random

print("Sınav Kağıtları Hazırlanıyor (Eğitim Dışı Verilerden)...")

# Asistan sadece ilk 1600 hastayı görmüştü. Biz 2000. sıradan sonrasına gidiyoruz!
hic_gorulmeden_kalanlar = tam_hasta_listesi[2000:]

# 3 Adet Rastgele Kanamalı Hasta Bulalım
kanamali_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
kanamali_havuz = kanamali_havuz[kanamali_havuz['any'] == 1].index.tolist()
sinav_kanamalilar = random.sample(kanamali_havuz, 3)

# 2 Adet Rastgele Sapasağlam Hasta Bulalım
saglam_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
saglam_havuz = saglam_havuz[saglam_havuz['any'] == 0].index.tolist()
sinav_saglamlar = random.sample(saglam_havuz, 2)

# Sınav Dosyasını Birleştirip Karıştıralım ki Kopya Çekemesin!
sinav_hastalar = sinav_kanamalilar + sinav_saglamlar
random.shuffle(sinav_hastalar) 

print("Sınav Başladı! 5 Hastanın Tomografisi İnceleniyor...\n")

kanama_tipleri = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

# 5 Hastayı yan yana göstermek için geniş bir tablo çiziyoruz
fig, axes = plt.subplots(1, 5, figsize=(25, 7))

for i, hasta_id in enumerate(sinav_hastalar):
    gercek_tanilar = hasta_matrisi.loc[hasta_id]
    hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + hasta_id + ".dcm"
    
    # Pencereleri Giydirme
    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]
    _, _, inter, slope = [get_first(x) for x in fields]

    img = dataset.pixel_array.astype(float)
    red = window_image(img.copy(), 40, 80, inter, slope)
    green = window_image(img.copy(), 80, 200, inter, slope)
    blue = window_image(img.copy(), 600, 2800, inter, slope)

    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
    
    # 256x256 modele uygun küçültme
    rgb_resized = cv2.resize(rgb, (256, 256))
    yapay_zeka_verisi = np.expand_dims(rgb_resized, axis=0)
    
    # ================================
    # ASİSTAN CEVAPLIYOR (Predict)
    tahmin_yuzdeleri = bizim_model.predict(yapay_zeka_verisi, verbose=0)[0]
    # ================================

    # Çıktıları Yazdırma
    ax = axes[i]
    ax.imshow(rgb)
    ax.axis('off')
    
    gercek_any = "KANAMA VAR" if gercek_tanilar['any'] == 1 else "SAGLIKLI"
    
    # Tıbbi Sınav Raporu
    rapor = f"Hasta: {hasta_id[-8:]}\n"
    rapor += f"👨‍⚕️ UZMAN: {gercek_any}\n"
    rapor += '-'*28 + '\n'
    rapor += "TUR    | UZMAN | ASİSTAN\n"
    
    for idx, tip in enumerate(kanama_tipleri):
        g = "VAR " if gercek_tanilar[tip] == 1 else "YOK "
        t_yuzde = tahmin_yuzdeleri[idx] * 100
        # Asistanın kararı %50 üzerindeyse tehlike var demektir
        t_karar = "🚨" if t_yuzde > 50 else "✅"
        
        rapor += f"{tip[:6].upper().ljust(6)} | {g}  | %{t_yuzde:02.0f} {t_karar}\n"
        
    ax.set_title(rapor, loc='left', fontsize=12, fontfamily='monospace')

plt.tight_layout()
plt.show()


In [ ]:
import random

print("Başhekimin Emriyle Sınıf Dengesizliği (Class Imbalance / Hile) Çözülüyor...")

# 1. Kanamalı ve Sağlam hastalar havuzunu tamamen ikiye ayıralım
kanamali_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 1].index.tolist()
saglikli_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 0].index.tolist()

# 2. "Şok Sınavı" Dosyası (Tam 800 Kanamalı + 800 Sağlıklı = Yarı Yarıya Dağılım)
dengeli_hastalar = kanamali_havuz_tam[:800] + saglikli_havuz_tam[:800]

# Asistan kalıbı ezberlemesin diye kağıtların sırasını iyice karıştıralım
random.shuffle(dengeli_hastalar)

# 3. Yürüyen Bandı bu "Torpilsiz ve Dengeli" listeyle yeniden kuralım
dengeli_egitim_bandi = RSNADataset(dengeli_hastalar, hasta_matrisi, batch_size=32)

print("\nKoli İçeriği Düzeltildi! Bantta Artık Yarı Yarıya Hasta Var.")
print("Asistanın Beynindeki Eski Temel Tembellikleri Silip, Eğitime Alıyoruz...")

# ----- YENİ VE TEMİZ BİR BEYİN -----
x = temel_beyin.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
kesin_tanilar = tf.keras.layers.Dense(6, activation='sigmoid')(x)

yeni_asistan = tf.keras.models.Model(inputs=temel_beyin.input, outputs=kesin_tanilar)
yeni_asistan.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), 
                     loss='binary_crossentropy', 
                     metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)])
# -----------------------------------

# Şimdi Dengeli Hileyi Kırma Eğitimi Başlasın (Test adımı belirtmiyoruz çünkü listeyi tam 1600 verdik zaten)
history_yeni = yeni_asistan.fit(
    dengeli_egitim_bandi, 
    epochs=5,
    verbose=1
)

# Eğitilen asistanı, asıl modelimizin üstüne kaydedelim ki ısı haritası bu düzelmiş modelle çalışsın
bizim_model = yeni_asistan

print("\nModelin tembelliği kırıldı! Asistan mecburen ter dökerek tüm kanamaları ayırt etmeyi çalıştı.")

In [ ]:
import random
import cv2
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("Şok Eğitiminden Çıkmış Asistanın Yeni '5 Hastalık Sınavı' Başlıyor...")

# Asistanın yeni dengeli eğitimde de görmediği havuzdan rastgele 5 yeni kişi (Unseen)
hic_gorulmeden_kalanlar = tam_hasta_listesi[2000:]

kanamali_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
kanamali_havuz = kanamali_havuz[kanamali_havuz['any'] == 1].index.tolist()
sinav_kanamalilar = random.sample(kanamali_havuz, 3) # 3 Kanamalı Hasta

saglam_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
saglam_havuz = saglam_havuz[saglam_havuz['any'] == 0].index.tolist()
sinav_saglamlar = random.sample(saglam_havuz, 2) # 2 Sağlıklı Hasta

# Sınav Dosyasını Karıştır (Asistan kaç kişi sağlıklı bilmiyor)
sinav_hastalar = sinav_kanamalilar + sinav_saglamlar
random.shuffle(sinav_hastalar) 

kanama_tipleri = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

fig, axes = plt.subplots(1, 5, figsize=(25, 7))

for i, hasta_id in enumerate(sinav_hastalar):
    gercek_tanilar = hasta_matrisi.loc[hasta_id]
    hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + hasta_id + ".dcm"
    
    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]
    
    def get_first_local(x):
        if type(x) == pydicom.multival.MultiValue: return int(x[0])
        return int(x)

    _, _, inter, slope = [get_first_local(x) for x in fields]

    img = dataset.pixel_array.astype(float)
    red = window_image(img.copy(), 40, 80, inter, slope)
    green = window_image(img.copy(), 80, 200, inter, slope)
    blue = window_image(img.copy(), 600, 2800, inter, slope)

    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
    
    rgb_resized = cv2.resize(rgb, (256, 256))
    yapay_zeka_verisi = np.expand_dims(rgb_resized, axis=0)
    
    # ====================================================
    # YENİ (DENGELİ) MODEL İLE TAHMİN
    tahmin_yuzdeleri = bizim_model.predict(yapay_zeka_verisi, verbose=0)[0]
    # ====================================================
    
    ax = axes[i]
    ax.imshow(rgb)
    ax.axis('off')
    
    gercek_any = "KANAMALI" if gercek_tanilar['any'] == 1 else "SAGLIKLI"
    
    rapor = f"Hasta: {hasta_id[-8:]}\n"
    rapor += f"👨‍⚕️ UZMAN: {gercek_any}\n"
    rapor += '-'*28 + '\n'
    rapor += "TUR    | UZMAN | ASİSTAN\n"
    
    for idx, tip in enumerate(kanama_tipleri):
        g = "VAR " if gercek_tanilar[tip] == 1 else "YOK "
        t_yuzde = tahmin_yuzdeleri[idx] * 100
        
        # EĞER ASİSTAN %50'DEN ŞÜPHELİYSE ALARM (🚨), EMİNSE (✅) VERSİN
        t_karar = "🚨" if t_yuzde >= 50 else "✅"
        
        rapor += f"{tip[:6].upper().ljust(6)} | {g}  | %{t_yuzde:02.0f} {t_karar}\n"
        
    ax.set_title(rapor, loc='left', fontsize=12, fontfamily='monospace')

plt.tight_layout()
plt.show()


In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pydicom

print("Tıbbi Anatomik (Isı Haritası / Grad-CAM) Analizi Başlıyor...\n")

# Keras 3'ün otomatik arama bug'ına takılmamak için, 
# EfficientNet'in vizyon merkezinin adını doğrudan sisteme kilitliyoruz.
son_gorsel_katman_adi = "top_activation"

print(f"Asistanın Göz Sinir Uçları Manuel Olarak Kilitlendi: {son_gorsel_katman_adi}")

def isi_haritasi_cikar(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        # Karar verilen en güçlü hastalık skorunu hedef al
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Test için meşhur kanamalı hastamızı alalım
ornek_hasta_id = kanamali_hastalar.index[0]
hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + ornek_hasta_id + ".dcm"

# Görüntüyü Sizin Profesyonel Pencerelerinizden Geçirelim
dataset = pydicom.dcmread(hedef_dosya)
fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
          dataset[('0028','1052')].value, dataset[('0028','1053')].value]

def get_first_local(x):
    if type(x) == pydicom.multival.MultiValue: return int(x[0])
    return int(x)

_, _, inter, slope = [get_first_local(x) for x in fields]

img = dataset.pixel_array.astype(float)
red = window_image(img.copy(), 40, 80, inter, slope)
green = window_image(img.copy(), 80, 200, inter, slope)
blue = window_image(img.copy(), 600, 2800, inter, slope)

rgb = np.zeros((img.shape[0], img.shape[1], 3))
rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

rgb_resized = cv2.resize(rgb, (256, 256))
img_array = np.expand_dims(rgb_resized, axis=0)

# ---------------------------------------------
# İŞTE MUCİZE BURADA: Asistanı Isı Haritasına Zorla!
heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)
# ---------------------------------------------

# Haritayı Çizilebilir Formata Getirme (Kırmızı


In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pydicom

print("Tıbbi Anatomik (Isı Haritası / Grad-CAM) Analizi Başlıyor...\n")

# Arama işlemini iptal edip, Asistanın şüpheyi oluşturduğu asıl Göz Sinirini (top_activation) doğrudan veriyoruz:
son_gorsel_katman_adi = "top_activation"

def isi_haritasi_cikar(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        # Asistanın en çok şüphelendiği (Örn: Subdural) noktaya odaklan!
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# ---------------------------------------------
# Test edilecek Kanamalı Hastamız
ornek_hasta_id = kanamali_hastalar.index[0]
hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + ornek_hasta_id + ".dcm"

# 1. CT Görüntüsünü Tıbbi Pencerelere Ayır
dataset = pydicom.dcmread(hedef_dosya)
fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
          dataset[('0028','1052')].value, dataset[('0028','1053')].value]

def get_first_local(x):
    if type(x) == pydicom.multival.MultiValue: return int(x[0])
    return int(x)

_, _, inter, slope = [get_first_local(x) for x in fields]

img = dataset.pixel_array.astype(float)
red = window_image(img.copy(), 40, 80, inter, slope)
green = window_image(img.copy(), 80, 200, inter, slope)
blue = window_image(img.copy(), 600, 2800, inter, slope)

rgb = np.zeros((img.shape[0], img.shape[1], 3))
rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

rgb_resized = cv2.resize(rgb, (256, 256))
img_array = np.expand_dims(rgb_resized, axis=0)

# 2. Isı Haritasını Çıkar
heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)

# 3. YZ Odak Bölgesini Normal Görüntüye "Alev (Jet)" Renginde Monte Et
heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
heatmap_resized = np.uint8(255 * heatmap_resized)
jet = plt.colormaps.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap_resized]

superimposed_img = jet_heatmap * 0.4 + rgb
superimposed_img = np.clip(superimposed_img, 0, 1)

# Ekrana Çiz!
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title(f"A) {ornek_hasta_id} ORİJİNAL KESİT (UZMAN)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(superimposed_img)
plt.title("B) ASİSTANIN KANAMA ŞÜPHESİ İÇİN BAKTIĞI DOKU (Grad-CAM)")
plt.axis('off')

plt.tight_layout()
plt.show()

print("\n🚀 Başhekimim (Clinical Lead), sağdaki görüntüde yapay zeka anatomi olarak kafa derisine mi yığılmış yoksa kanama bölgesine mi bakıyor?")


In [ ]:
print("Müdahale Başlıyor: Asistanın Görsel Korteksi Kilitleniyor (Freezing)...")

# 1. Asistanın şekilleri ezberlemesini önlemek için temel beynini KİLİTLİYORUZ!
# Artık kafatasını ezberleyemez, sadece teşhis koymayı öğrenmek ZORUNDA.
temel_beyin.trainable = False

# 2. Üzerine yepyeni, tertemiz bir Karar Lobu ekleyelim
x_yeni = temel_beyin.output
x_yeni = tf.keras.layers.GlobalAveragePooling2D()(x_yeni)
x_yeni = tf.keras.layers.Dropout(0.4)(x_yeni) # Kafası karışmasın diye unutma/eleme oranını %40'a çıkardık
kesin_tanilar_yeni = tf.keras.layers.Dense(6, activation='sigmoid')(x_yeni)

# Cerrah işlemi tamam
iyilestirilmis_model = tf.keras.models.Model(inputs=temel_beyin.input, outputs=kesin_tanilar_yeni)

# Sistemi Derleyelim (Öğrenme hızını normal standartlara aldık)
iyilestirilmis_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                             loss='binary_crossentropy', 
                             metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)])

print("🧠 Beyin Donduruldu! Asistan çok daha zorlu ve kopyasız bir eğitime alınıyor...")

# Öğrenme süreci (Beyin kilitli olduğu için GPU çok hızlı işleyecek)
history_iyilestirilmis = iyilestirilmis_model.fit(
    dengeli_egitim_bandi, 
    epochs=10,
    verbose=1
)

# Eğitilen iyileştirilmiş asistanı asıl modelimize eşitleyelim 
bizim_model = iyilestirilmis_model

print("\n🎯 Tedavi (İyileştirme) Tamamlandı! Artık ısı haritasını tekrar test edebiliriz.")


In [ ]:
import random
import cv2
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("🧠 Dondurulmuş ve İyileştirilmiş Asistanın 'Yeni 5 Hastalık Çapraz Sorgusu' Başlıyor...\n")

# Şüphe kalmasın diye yine tamamen YEPYENİ 5 kişi (Unseen) rastgele çağrılıyor
hic_gorulmeden_kalanlar = tam_hasta_listesi[2000:]

kanamali_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
kanamali_havuz = kanamali_havuz[kanamali_havuz['any'] == 1].index.tolist()
sinav_kanamalilar = random.sample(kanamali_havuz, 3) 

saglam_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
saglam_havuz = saglam_havuz[saglam_havuz['any'] == 0].index.tolist()
sinav_saglamlar = random.sample(saglam_havuz, 2) 

sinav_hastalar = sinav_kanamalilar + sinav_saglamlar
random.shuffle(sinav_hastalar) 

kanama_tipleri = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

fig, axes = plt.subplots(1, 5, figsize=(25, 7))

for i, hasta_id in enumerate(sinav_hastalar):
    gercek_tanilar = hasta_matrisi.loc[hasta_id]
    hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + hasta_id + ".dcm"
    
    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]
    
    def get_first_local(x):
        if type(x) == pydicom.multival.MultiValue: return int(x[0])
        return int(x)

    _, _, inter, slope = [get_first_local(x) for x in fields]

    img = dataset.pixel_array.astype(float)
    red = window_image(img.copy(), 40, 80, inter, slope)
    green = window_image(img.copy(), 80, 200, inter, slope)
    blue = window_image(img.copy(), 600, 2800, inter, slope)

    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
    
    rgb_resized = cv2.resize(rgb, (256, 256))
    yapay_zeka_verisi = np.expand_dims(rgb_resized, axis=0)
    
    # ====================================================
    # BEYNİ DONDURULMUŞ YENİ MODEL İLE TAHMİN
    tahmin_yuzdeleri = bizim_model.predict(yapay_zeka_verisi, verbose=0)[0]
    # ====================================================
    
    ax = axes[i]
    ax.imshow(rgb)
    ax.axis('off')
    
    gercek_any = "KANAMALI" if gercek_tanilar['any'] == 1 else "SAGLIKLI"
    
    rapor = f"Hasta: {hasta_id[-8:]}\n"
    rapor += f"👨‍⚕️ UZMAN: {gercek_any}\n"
    rapor += '-'*28 + '\n'
    rapor += "TUR    | UZMAN | ASİSTAN\n"
    
    for idx, tip in enumerate(kanama_tipleri):
        g = "VAR " if gercek_tanilar[tip] == 1 else "YOK "
        t_yuzde = tahmin_yuzdeleri[idx] * 100
        
        # Asistan artık en az %40 şüphe duyarsa uyarı mekanizmasını tetiklesin
        t_karar = "🚨" if t_yuzde >= 40 else "✅"  
        
        rapor += f"{tip[:6].upper().ljust(6)} | {g}  | %{t_yuzde:02.0f} {t_karar}\n"
        
    ax.set_title(rapor, loc='left', fontsize=12, fontfamily='monospace')

plt.tight_layout()
plt.show()


In [ ]:
import random

print("🏥 Veri Havuzu Genişletiliyor (1.600 Hastadan -> 8.000 Hastaya Çıkış)...\n")

# Kanamalı ve Sağlam hastalar havuzu
kanamali_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 1].index.tolist()
saglikli_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 0].index.tolist()

# 4000 Kanamalı ve 4000 Sağlıklı olmak üzere veri setini tam 5 KAT BÜYÜTÜYORUZ!
dev_dengeli_hastalar = kanamali_havuz_tam[:4000] + saglikli_havuz_tam[:4000]

# Tembellik yapıp sırayı ezberlemesin diye listeyi iyice karıştırıyoruz
random.shuffle(dev_dengeli_hastalar)

# Yeni "Büyük Kapasiteli" Yürüyen Bant
dev_egitim_bandi = RSNADataset(dev_dengeli_hastalar, hasta_matrisi, batch_size=32)

print("Kapasite Artırıldı! Asistan bu 8.000 hastalık dev arşivi incelemeye başlıyor...")

# Zaten önceki adımda Dondurduğumuz modelimizi (iyilestirilmis_model) 
# bu devasa veri yığınıyla eğitiyoruz (Bu kez tur sayısı uzun süreceği için sadece 5 Tur dönüyoruz)
history_dev = iyilestirilmis_model.fit(
    dev_egitim_bandi, 
    epochs=5,
    verbose=1
)

# Sistemimizi güncelleyelim
bizim_model = iyilestirilmis_model

print("\n✅ Harika! Asistan artık 1.600 değil, 8.000 hastanın paha biçilemez tecrübesine sahip!")


In [ ]:
import random
import cv2
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("📊 8.000 Hastalık Tecrübeye Çıkan Asistanın 5 Hastalık Çapraz Sorgusu Başlıyor...\n")

# Şüphe kalmasın diye yine tamamen YEPYENİ 5 kişi (Unseen) rastgele çağrılıyor
# Asistanın eğitimde denk gelmiş olma ihtimaline karşı testi listenin çok daha ilerisinden alıyoruz (5000'den sonrası)
hic_gorulmeden_kalanlar = tam_hasta_listesi[5000:]

kanamali_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
kanamali_havuz = kanamali_havuz[kanamali_havuz['any'] == 1].index.tolist()
sinav_kanamalilar = random.sample(kanamali_havuz, 3) 

saglam_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
saglam_havuz = saglam_havuz[saglam_havuz['any'] == 0].index.tolist()
sinav_saglamlar = random.sample(saglam_havuz, 2) 

sinav_hastalar = sinav_kanamalilar + sinav_saglamlar
random.shuffle(sinav_hastalar) 

kanama_tipleri = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

fig, axes = plt.subplots(1, 5, figsize=(25, 7))

for i, hasta_id in enumerate(sinav_hastalar):
    gercek_tanilar = hasta_matrisi.loc[hasta_id]
    hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + hasta_id + ".dcm"
    
    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]
    
    def get_first_local(x):
        if type(x) == pydicom.multival.MultiValue: return int(x[0])
        return int(x)

    _, _, inter, slope = [get_first_local(x) for x in fields]

    img = dataset.pixel_array.astype(float)
    red = window_image(img.copy(), 40, 80, inter, slope)
    green = window_image(img.copy(), 80, 200, inter, slope)
    blue = window_image(img.copy(), 600, 2800, inter, slope)

    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
    
    rgb_resized = cv2.resize(rgb, (256, 256))
    yapay_zeka_verisi = np.expand_dims(rgb_resized, axis=0)
    
    # ====================================================
    # SADECE VERİSİ 8.000'E ÇIKARTILMIŞ MODEL İLE TAHMİN
    tahmin_yuzdeleri = bizim_model.predict(yapay_zeka_verisi, verbose=0)[0]
    # ====================================================
    
    ax = axes[i]
    ax.imshow(rgb)
    ax.axis('off')
    
    gercek_any = "KANAMALI" if gercek_tanilar['any'] == 1 else "SAGLIKLI"
    
    rapor = f"Hasta: {hasta_id[-8:]}\n"
    rapor += f"👨‍⚕️ UZMAN: {gercek_any}\n"
    rapor += '-'*28 + '\n'
    rapor += "TUR    | UZMAN | ASİSTAN\n"
    
    for idx, tip in enumerate(kanama_tipleri):
        g = "VAR " if gercek_tanilar[tip] == 1 else "YOK "
        t_yuzde = tahmin_yuzdeleri[idx] * 100
        
        # Asistan artık en az %40 şüphe duyarsa alarmı (🚨) tetikleyecek
        t_karar = "🚨" if t_yuzde >= 40 else "✅"  
        
        rapor += f"{tip[:6].upper().ljust(6)} | {g}  | %{t_yuzde:02.0f} {t_karar}\n"
        
    ax.set_title(rapor, loc='left', fontsize=12, fontfamily='monospace')

plt.tight_layout()
plt.show()


In [ ]:
import random

print("📈 Veri Havuzu Zirveye Çıkıyor (8.000 Hastadan -> 20.000 Hastaya)...\n")

# Kanamalı ve Sağlam hastalar havuzumuzu yeniden alıyoruz
kanamali_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 1].index.tolist()
saglikli_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 0].index.tolist()

# 10.000 Kanamalı + 10.000 Sağlıklı = 20.000 Hastalık MEGA Veri Seti!
# Asistanın görüş açısını inanılmaz düzeyde artırıyoruz.
mega_dengeli_hastalar = kanamali_havuz_tam[:10000] + saglikli_havuz_tam[:10000]

random.shuffle(mega_dengeli_hastalar)

# Yeni "Mega Kapasiteli" Yürüyen Bant
mega_egitim_bandi = RSNADataset(mega_dengeli_hastalar, hasta_matrisi, batch_size=32)

print("20.000 Hastalık Mega Veri Bandı Kuruldu!")
print("Sabrınızın karşılığını alacaksınız... Kutu sayısı çok arttığı için bu aşama biraz vakit alacaktır.")

# Zaten donuk olan (sadece karar lobu açık) iyilestirilmis_modelimizi bu dev yığınla eğitiyoruz
history_mega = iyilestirilmis_model.fit(
    mega_egitim_bandi, 
    epochs=5,
    verbose=1
)

# Başarılarımızı asıl sisteme kaydedelim
bizim_model = iyilestirilmis_model

print("\n🏆 İnanılmaz! Asistan şu an 20.000 farklı hastanın tomografisini incelemiş gerçek bir 'Uzman Adayı' (Fellow) seviyesine çıktı! \nO beklettiğiniz 5'li görsel taramayı tekrar yapıp farkı görebilirsiniz.")


### 🎯 Stage 2 Training: Fine-Tuning for Microscopic Textural Vision
After processing 20,000 cases, the Frozen model encounters a Representational Bottleneck (plateauing at ~70% AUC). To elevate the machine's predictive power to an Expert level, the top 20 layers of the visual cortex are **Unfrozen**. The learning rate is drastically lowered to `1e-4`, allowing the network to meticulously adapt its weights to the microscopic textures of blood clots without suffering from catastrophic forgetting.

In [ ]:
import tensorflow as tf

print("Mükemmeliyetçi Son Aşama: İnce Ayar (Fine-Tuning) Başlıyor...\n")

# 1. Asistanın göz yeteneğindeki prangaları çözüyoruz (Unfreeze)
temel_beyin.trainable = True

# 2. Tamamını çözersek bu sefer eski zekasını da kaybeder (Amnesia) diye,
# SADECE teşhisleri okuyan son 20 sinir ucunu serbest bırakıyoruz.
for layer in temel_beyin.layers[:-20]:
    layer.trainable = False

# 3. Öğrenme hızını İNANILMAZ TÖRPÜLÜYORUZ (0.001 -> 0.0001)
# Son 20 sinir ucunda "Mikroskobik" bir cerrahi ameliyat yapıyoruz, yavaş ve emin adımlarla gidecek.
iyilestirilmis_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
                             loss='binary_crossentropy', 
                             metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)])

print("🧠 Zincirler Kırıldı! Asistan kanın mikroskobik dokusunu (Texture) kavramaya çalışıyor...")

# Sadece 5 tur ilerliyoruz çünkü ince ayarda fazla gezmek her şeyi bozabilir
history_finetune = iyilestirilmis_model.fit(
    mega_egitim_bandi, # O yorucu 20.000'lik devasa tesis
    epochs=5,
    verbose=1
)

# Eğitilen şaheseri asıl sistem otomasyonuna bağlayalım
bizim_model = iyilestirilmis_model

print("\n🔥 BİLGİSAYARLI GÖRÜNÜN (COMPUTER VISION) ZİRVESİNE ULAŞILDI: FINE-TUNING BİTTİ!")


### 📊 Radiological Board Exam: A Clinical Audit
Relying solely on mathematical Accuracy metrics can be clinically deceiving. For a robust audit, **10 completely unseen patients** were randomly sampled for a visual 'Board Exam', pitting the AI's deductions against the Ground Truth of an Expert Radiologist. While the AI successfully isolated true hemorrhages, it demonstrated a critical clinical vulnerability: triggering **False Positives** on physiological Choroid Plexus/Pineal calcifications and beam-hardening bone artifacts. This highlights the indispensable necessity of human oversight in Medical AI.

In [ ]:
import random
import cv2
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("📊 10 Hastalık 'Board Sınavı' (Kapsamlı Test) Başlıyor...\n")

# Hile yapamaması için listenin en sonundaki (hiç görmediği) hastalara gidiyoruz
hic_gorulmeden_kalanlar = tam_hasta_listesi[10000:]

kanamali_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
kanamali_havuz = kanamali_havuz[kanamali_havuz['any'] == 1].index.tolist()
sinav_kanamalilar = random.sample(kanamali_havuz, 5) # 5 Kanamalı Kesit

saglam_havuz = hasta_matrisi.loc[hic_gorulmeden_kalanlar]
saglam_havuz = saglam_havuz[saglam_havuz['any'] == 0].index.tolist()
sinav_saglamlar = random.sample(saglam_havuz, 5) # 5 Sağlıklı Kesit

sinav_hastalar = sinav_kanamalilar + sinav_saglamlar
random.shuffle(sinav_hastalar) 

kanama_tipleri = ['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

# 2 Satır, 5 Sütunculuk dev bir ekran
fig, axes = plt.subplots(2, 5, figsize=(25, 12))
axes = axes.flatten()

for i, hasta_id in enumerate(sinav_hastalar):
    gercek_tanilar = hasta_matrisi.loc[hasta_id]
    hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + hasta_id + ".dcm"
    
    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]
    
    def get_first_local(x):
        if type(x) == pydicom.multival.MultiValue: return int(x[0])
        return int(x)

    _, _, inter, slope = [get_first_local(x) for x in fields]

    img = dataset.pixel_array.astype(float)
    red = window_image(img.copy(), 40, 80, inter, slope)
    green = window_image(img.copy(), 80, 200, inter, slope)
    blue = window_image(img.copy(), 600, 2800, inter, slope)

    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
    
    rgb_resized = cv2.resize(rgb, (256, 256))
    yapay_zeka_verisi = np.expand_dims(rgb_resized, axis=0)
    
    # NİHAİ (FINE-TUNED) MODEL İLE TAHMİN
    tahmin_yuzdeleri = bizim_model.predict(yapay_zeka_verisi, verbose=0)[0]
    
    ax = axes[i]
    ax.imshow(rgb)
    ax.axis('off')
    
    gercek_any = "KANAMALI" if gercek_tanilar['any'] == 1 else "SAGLIKLI"
    
    rapor = f"Hasta: {hasta_id[-8:]}\n"
    rapor += f"👨‍⚕️ UZMAN: {gercek_any}\n"
    rapor += '-'*25 + '\n'
    rapor += "TUR   | UZ | ASİSTAN\n"
    
    for idx, tip in enumerate(kanama_tipleri):
        g = "VAR" if gercek_tanilar[tip] == 1 else "YOK"
        t_yuzde = tahmin_yuzdeleri[idx] * 100
        
        # Eğer kanama ihtimali %40 ve üzeriyse Kırmızı Alarm (🚨) versin
        t_karar = "🚨" if t_yuzde >= 40 else "✅"  
        
        # Ekran dar olacağı için yazıları hafif kısalttık
        rapor += f"{tip[:5].upper().ljust(5)} | {g} | %{t_yuzde:02.0f} {t_karar}\n"
        
    ax.set_title(rapor, loc='left', fontsize=11, fontfamily='monospace')

plt.tight_layout()
plt.show()

### 🔪 Surgical Intervention: Skull-Stripping & The "Domain Shift" Shock
To medically resolve the AI's vulnerability to bone artifacts, the CT images were subjected to a computer vision masking process based on Hounsfield Unit (HU) density thresholds prior to model inference. The skull was completely stripped, leaving only the brain parenchyma.
**Clinical Finding:** Even with the skull fully removed, the AI could not drop its suspicion to 0%. This confirms a vital machine learning phenomenon: **Domain Shift Shock**. Because the model acquired its 20,000 experiences solely observing "skulls wrapped around brains", presenting it with a "naked skull-stripped" brain caused spatial disorientation. *This elegantly proves that radiological preprocessing must be integrated during the mass training phase, not just inserted as a band-aid during inference.*

In [ ]:
import cv2
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("🧠 RADYOLOJİK CERRAHİ: Kafatası Soyma (Skull-Stripping) İşlemi Başlıyor...\n")

def kafatasi_soy_ct(rgb_resim):
    # Görüntümüzü [0-255] piksel aralığında sabitleyelim
    rgb_255 = np.uint8(rgb_resim * 255) if np.max(rgb_resim) <= 1 else np.uint8(rgb_resim)
    
    # Brain Window (Beyin Penceresi) RGB sistemimizde Kırmızı (İlk) kanaldadır
    # Kafatası kemiği bu pencerede en parlak (255) piksellerdir. 
    gray = rgb_255[:,:,0] 
    
    # 1. Kemiği Kesin Olarak Tespit Et (Çok parlak pikseller)
    _, kemik_maskesi = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
    
    # 2. Kemik hattındaki boşlukları kapat (Morfolojik Kapanma / Closing)
    kernel = np.ones((5,5), np.uint8)
    kemik_maskesi = cv2.morphologyEx(kemik_maskesi, cv2.MORPH_CLOSE, kernel)
    
    # 3. Kafatasının Çember Sınırlarını Bul
    contours, _ = cv2.findContours(kemik_maskesi, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    
    beyin_maskesi = np.zeros(gray.shape, dtype=np.uint8)
    
    if len(contours) > 0:
        # En büyük dış sınır kafatasının en dış hattıdır
        en_buyuk_kontur = max(range(len(contours)), key=lambda i: cv2.contourArea(contours[i]))
        
        # O sınırın İÇİNİ (Tüm kafayı) Maskeyle Doldur
        cv2.drawContours(beyin_maskesi, contours, en_buyuk_kontur, 255, -1)
        
        # ŞİMDİ EN KRİTİK NOKTA: Maskeden kemiğin (kafa tası zarının) KENDİSİNİ ÇIKAR!
        # Böylece sadece beynin iç yumuşak dokusu maskelenmiş kalır.
        beyin_maskesi = cv2.subtract(beyin_maskesi, kemik_maskesi)
        
        # Kemiğin ince kenar / ışın saçılma (Beam Hardening) artefaktlarını temizlemek için maskeyi 2 tur aşındıralım (Erosion)
        beyin_maskesi = cv2.erode(beyin_maskesi, kernel, iterations=2)
    
    # Çıkarılan bu muazzam maskeyi 3-kanallı resme uygulaıp etrafı karanlığa gömelim
    beyin_maskesi_3d = np.stack([beyin_maskesi, beyin_maskesi, beyin_maskesi], axis=2) / 255.0
    soyulmus_resim = rgb_resim * beyin_maskesi_3d
    
    return soyulmus_resim, beyin_maskesi

# Az önce Asistanın KEMİĞİ kanama sandığı 2. Yalancı Pozitif hastamızı (İndeks 1) getirelim
hatali_hasta_id = sinav_hastalar[1]
hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + hatali_hasta_id + ".dcm"

# CT Oku ve Pencerele (Standart)
dataset = pydicom.dcmread(hedef_dosya)
fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
          dataset[('0028','1052')].value, dataset[('0028','1053')].value]

def get_first_local(x):
    if type(x) == pydicom.multival.MultiValue: return int(x[0])
    return int(x)

_, _, inter, slope = [get_first_local(x) for x in fields]
img = dataset.pixel_array.astype(float)

red = window_image(img.copy(), 40, 80, inter, slope)
green = window_image(img.copy(), 80, 200, inter, slope)
blue = window_image(img.copy(), 600, 2800, inter, slope)
rgb = np.zeros((img.shape[0], img.shape[1], 3))
rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
rgb_resized = cv2.resize(rgb, (256, 256))

# BÜYÜ MÜDAHALE: Kafatasını Piksellerden Sök
soyulmus_rgb, maske = kafatasi_soy_ct(rgb_resized)

# GÖRSELLEŞTİRME
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(rgb_resized)
plt.title("A) ORİJİNAL KESİT\n(Sol taraftaki kalın kemik artefaktına dikkat)")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(maske, cmap='gray')
plt.title("B) MATEMATİKSEL BEYİN MASKESİ\n(Kafatası silindi)")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(soyulmus_rgb)
plt.title("C) KAFATASI SOYULMUŞ YENİ RÖNTGEN\n(Artık sadece kanamaya bakılabilir)")
plt.axis('off')

plt.tight_layout()
plt.show()

# Bakalım asistan "Kemiksiz" kalınca aynı yalanı atabilecek mi?
temiz_veri = np.expand_dims(soyulmus_rgb, axis=0)
eski_tahmin = bizim_model.predict(np.expand_dims(rgb_resized, axis=0), verbose=0)[0][0] * 100
yeni_tahmin = bizim_model.predict(temiz_veri, verbose=0)[0][0] * 100

print(f"\n🩺 Asistanın Orijinal Kesitteki Evham Şüphesi: % {eski_tahmin:.1f} 🚨")
print(f"🩺 TIRAŞLI Kesite Bakınca Asistanın Şüphesi: % {yeni_tahmin:.1f} ✅")


### 🕵️ Medical Autopsy: Anatomical Illusion Map (Error-Focused Grad-CAM)
In standard Computer Vision, Heatmaps are typically used to boast correct predictions. In this study, guided by radiological scrutiny, Grad-CAM is utilized as an "Autopsy" tool to decipher **where and why the machine failed**. The heatmaps anatomically prove that the model wasn't hallucinating blood out of thin air; rather, it was misled by physiological calcifications and thick asymmetric bone artifacts, establishing a *Spurious Correlation* between high pixel density and pathology.

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pydicom

print("🕵️ OTOPSİ BAŞLIYOR: Tıbbi Yanılsama Haritalarının (Error-Focused Grad-CAM) Deşifresi...\n")

son_gorsel_katman_adi = "top_activation"

def isi_haritasi_cikar(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        # Asistanın o hatalı kararını verirken "En çok şüphelendiği" noktaya kitlen
        pred_index = tf.argmax(preds[0]) 
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Sınavda kandırılan iki sağlıklı hastamızı hafızadan çekiyoruz
# 2. Hasta (Kemiğe aldanan) -> İndeks 1
# 4. Hasta (Kireçlenmeye aldanan) -> İndeks 3
tuzağa_dusenler = [sinav_hastalar[1], sinav_hastalar[3]]
hata_tipleri = ["KAFATASI ARTEFAKTI (Kemiğin Parlaması)", "KALSİFİKASYON (Kireçlenme)"]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

for i, y_id in enumerate(tuzağa_dusenler):
    hedef_dosya = "/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/" + y_id + ".dcm"

    dataset = pydicom.dcmread(hedef_dosya)
    fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
              dataset[('0028','1052')].value, dataset[('0028','1053')].value]

    def get_first_local(x):
        if type(x) == pydicom.multival.MultiValue: return int(x[0])
        return int(x)

    _, _, inter, slope = [get_first_local(x) for x in fields]

    img = dataset.pixel_array.astype(float)
    red = window_image(img.copy(), 40, 80, inter, slope)
    green = window_image(img.copy(), 80, 200, inter, slope)
    blue = window_image(img.copy(), 600, 2800, inter, slope)

    rgb = np.zeros((img.shape[0], img.shape[1], 3))
    rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

    rgb_resized = cv2.resize(rgb, (256, 256))
    img_array = np.expand_dims(rgb_resized, axis=0)

    # ---------------------------------------------
    # Asistanın o %88 ve %73 uydurma şüphesi nereden geliyor? Isı Haritasını alalım.
    # ---------------------------------------------
    heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)
    heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    jet = plt.colormaps.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]

    superimposed_img = jet_heatmap * 0.4 + rgb
    superimposed_img = np.clip(superimposed_img, 0, 1)

    tahmin_orani = bizim_model.predict(img_array, verbose=0)[0][0] * 100

    # Sol Sütun: Orijinal Çizim
    ax_orig = axes[i, 0]
    ax_orig.imshow(rgb)
    ax_orig.set_title(f"A) {y_id} (Gerçek: SAĞLIKLI)\nYZ'nın Yersiz Şüphesi: % {tahmin_orani:.1f} 🚨", fontsize=11)
    ax_orig.axis('off')

    # Sağ Sütun: YZ Yanılsaması (CSI Odak Çizimi)
    ax_heat = axes[i, 1]
    ax_heat.imshow(superimposed_img)
    ax_heat.set_title(f"B) YZ'NİN YANILDIĞI ODAK (Grad-CAM)\nTuzağa Düşüren Neden: {hata_tipleri[i]}", fontsize=11)
    ax_heat.axis('off')

plt.tight_layout()
plt.show()

print("\n👨‍⚕️ OTOPSİ SONUCU: Görsel kanıtlanmıştır! Yapay zekalar beyindeki anatomik tuzaklarda 'Radyolog Yönlendirmesine' muhtaçtır.")


---
# 🕵️‍♂️ Phase 3: Clinical Audit & Error Taxonomy (Grad-CAM Analysis)

From a clinical radiologist's perspective, achieving 85% or 90% accuracy is **not sufficient** for safe deployment in a medical setting. In real-world clinical scenarios, answering the question *"Why did the model fail?"* is more critical than the accuracy score itself.

To prevent the model from relying on "Shortcut Learning" or "Feature Biases" (e.g., confusing skull thickness with hemorrhage), we must systematically audit the cases where the model **failed**. 

### 📌 Audit Methodology: Hunting "False Positives"
In the pipeline below, we scan an unseen dataset of **healthy patients** (Ground Truth = 0). Our goal is to catch instances where the AI panicked and confidently predicted **"Hemorrhage Detected"** (Prediction > 50%).

Once we capture these False Positive cases, we will pass them through an XAI (Explainable AI) tool using **Grad-CAM** heatmaps. This will allow us to visualize whether the AI was deceived by **the skull boundary, physiological calcifications (e.g., choroid plexus), or beam hardening artifacts**, enabling us to build a structured clinical error taxonomy.

*The script below will run until it isolates and visualizes exactly 10 False Positive cases for our clinical review.*
---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pydicom
import random
import tensorflow as tf

print("🔎 CLINICAL AUDIT RADAR INITIALIZED!")
print("Scanning for 'False Positive' cases...\n")

# O hata veren "for" döngüsünü tamamen sildik. 
# EfficientNet'in görme merkezinin (son evrişim katmanının) adını biz zaten biliyoruz:
son_gorsel_katman_adi = "top_activation"

def isi_haritasi_cikar(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def get_first_local(x):
    if type(x) == pydicom.multival.MultiValue: return int(x[0])
    return int(x)

# Listelerimiz
test_patients = tam_hasta_listesi[2000:]
random.shuffle(test_patients)
captured_error_count = 0
TARGET_ERROR_COUNT = 10 

# Tarama başlıyor
for patient_id in test_patients:
    if captured_error_count >= TARGET_ERROR_COUNT:
        break 

    ground_truth = hasta_matrisi.loc[patient_id]['any']
    
    if ground_truth == 0:
        target_file = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{patient_id}.dcm"
        
        try:
            dataset = pydicom.dcmread(target_file)
            fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                      dataset[('0028','1052')].value, dataset[('0028','1053')].value]
            _, _, inter, slope = [get_first_local(x) for x in fields]

            img = dataset.pixel_array.astype(float)
            red = window_image(img.copy(), 40, 80, inter, slope)
            green = window_image(img.copy(), 80, 200, inter, slope)
            blue = window_image(img.copy(), 600, 2800, inter, slope)

            rgb = np.zeros((img.shape[0], img.shape[1], 3))
            rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

            rgb_resized = cv2.resize(rgb, (256, 256))
            img_array = np.expand_dims(rgb_resized, axis=0)
            
            predictions = bizim_model.predict(img_array, verbose=0)[0]
            any_pred_percentage = predictions[0] * 100 
            
            if any_pred_percentage > 50.0:
                captured_error_count += 1
                print(f"🚨 FALSE POSITIVE DETECTED: {patient_id} | Ground Truth: Healthy | AI Prediction: {any_pred_percentage:.1f}% Hemorrhage")
                
                heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)
                heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
                heatmap_resized = np.uint8(255 * heatmap_resized)
                jet = plt.colormaps.get_cmap("jet")
                jet_colors = jet(np.arange(256))[:, :3]
                jet_heatmap = jet_colors[heatmap_resized]

                superimposed_img = jet_heatmap * 0.4 + rgb
                superimposed_img = np.clip(superimposed_img, 0, 1)
                
                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                plt.imshow(rgb)
                plt.title(f"({captured_error_count}/{TARGET_ERROR_COUNT}) ORIGINAL CT")
                plt.axis('off')

                plt.subplot(1, 2, 2)
                plt.imshow(superimposed_img)
                plt.title("AI FOCUS MAP (GRAD-CAM)")
                plt.axis('off')

                # Görüntüleri Kaggle Output klasörüne kaydet
                save_path = f"/kaggle/working/FP_Case_{captured_error_count}_{patient_id}.png"
                plt.savefig(save_path, bbox_inches="tight", dpi=150)
                plt.close() 
                
        except Exception as e:
            print(f"❌ ERROR: {e}")

print("\n✅ Radar Scan Complete! 10 Images successfully saved to the Output (/kaggle/working) folder.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pydicom
import random
import tensorflow as tf

print("🔎 CALCIFICATION & ERROR SEARCH INITIALIZED!")
print("Targeting 50 False Positive cases to hunt for physiological calcification errors...\n")

son_gorsel_katman_adi = "top_activation"

def isi_haritasi_cikar(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def get_first_local(x):
    if type(x) == pydicom.multival.MultiValue: return int(x[0])
    return int(x)

test_patients = tam_hasta_listesi[2000:]
random.shuffle(test_patients)
captured_error_count = 0

# Hedefimizi 50 hastaya (yaklaşık 5-6 dakikalık işlem) çıkardık:
TARGET_ERROR_COUNT = 50 

for patient_id in test_patients:
    if captured_error_count >= TARGET_ERROR_COUNT:
        break 

    ground_truth = hasta_matrisi.loc[patient_id]['any']
    
    if ground_truth == 0:
        target_file = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{patient_id}.dcm"
        
        try:
            dataset = pydicom.dcmread(target_file)
            fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                      dataset[('0028','1052')].value, dataset[('0028','1053')].value]
            _, _, inter, slope = [get_first_local(x) for x in fields]

            img = dataset.pixel_array.astype(float)
            red = window_image(img.copy(), 40, 80, inter, slope)
            green = window_image(img.copy(), 80, 200, inter, slope)
            blue = window_image(img.copy(), 600, 2800, inter, slope)

            rgb = np.zeros((img.shape[0], img.shape[1], 3))
            rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

            rgb_resized = cv2.resize(rgb, (256, 256))
            img_array = np.expand_dims(rgb_resized, axis=0)
            
            predictions = bizim_model.predict(img_array, verbose=0)[0]
            any_pred_percentage = predictions[0] * 100 
            
            if any_pred_percentage > 50.0:
                captured_error_count += 1
                
                heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)
                heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
                heatmap_resized = np.uint8(255 * heatmap_resized)
                jet = plt.colormaps.get_cmap("jet")
                jet_colors = jet(np.arange(256))[:, :3]
                jet_heatmap = jet_colors[heatmap_resized]

                superimposed_img = jet_heatmap * 0.4 + rgb
                superimposed_img = np.clip(superimposed_img, 0, 1)
                
                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                plt.imshow(rgb)
                plt.title(f"({captured_error_count}/{TARGET_ERROR_COUNT}) ORIGINAL CT")
                plt.axis('off')

                plt.subplot(1, 2, 2)
                plt.imshow(superimposed_img)
                plt.title("AI FOCUS MAP (GRAD-CAM)")
                plt.axis('off')

                # İsimleri Calc_Search olarak değiştirdik
                save_path = f"/kaggle/working/Calc_Search_{captured_error_count}_{patient_id}.png"
                plt.savefig(save_path, bbox_inches="tight", dpi=150)
                plt.close() 
                print(f"🚨 False Positive #{captured_error_count} -> Saved to Output folder ({patient_id})")
                
        except Exception as e:
            pass

print("\n✅ Search Complete! 50 Images saved. Please check Output folder to hunt for Calcification Errors.")


In [ ]:
import numpy as np
import cv2
import pydicom
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("⚕️ KLİNİK PERFORMANS RAPORU (Confusion Matrix) HAZIRLANIYOR...")
print("Modelin hiç görmediği hastalar üzerindeki başarısı test ediliyor (Yaklaşık 1-2 dakika sürebilir)...\n")

# Görülmemiş hastalardan dengeli bir test grubu seçelim
test_hastalar = tam_hasta_listesi[2000:]

kanamali_havuz_test = hasta_matrisi.loc[test_hastalar]
kanamali_havuz_test = kanamali_havuz_test[kanamali_havuz_test['any'] == 1].index.tolist()

saglam_havuz_test = hasta_matrisi.loc[test_hastalar]
saglam_havuz_test = saglam_havuz_test[saglam_havuz_test['any'] == 0].index.tolist()

# Test için 100'er tane alalım (Hızlı sonuç görmek için)
test_kanamalilar = random.sample(kanamali_havuz_test, 100)
test_saglamlar = random.sample(saglam_havuz_test, 100)

test_grubu = test_kanamalilar + test_saglamlar
random.shuffle(test_grubu)

gercek_degerler = []
tahmin_edilenler = []

# Hastaları sırayla asistanın önüne (predict) gönderiyoruz
for i, hasta_id in enumerate(test_grubu):
    gercek_tani = hasta_matrisi.loc[hasta_id]['any']
    
    hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
    
    try:
        dataset = pydicom.dcmread(hedef_dosya)
        fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                  dataset[('0028','1052')].value, dataset[('0028','1053')].value]
        
        def get_first_local(x):
            if type(x) == pydicom.multival.MultiValue: return int(x[0])
            return int(x)

        _, _, inter, slope = [get_first_local(x) for x in fields]

        img = dataset.pixel_array.astype(float)
        red = window_image(img.copy(), 40, 80, inter, slope)
        green = window_image(img.copy(), 80, 200, inter, slope)
        blue = window_image(img.copy(), 600, 2800, inter, slope)

        rgb = np.zeros((img.shape[0], img.shape[1], 3))
        rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

        rgb_resized = cv2.resize(rgb, (256, 256))
        img_array = np.expand_dims(rgb_resized, axis=0)
        
        # Sadece "any" (herhangi bir kanama var mı?) sonucuna bakıyoruz
        prediction = bizim_model.predict(img_array, verbose=0)[0][0]
        
        # Yapay zeka %50'den fazla şüpheliyse kanama var (1), altındaysa yok (0) diyelim
        tahmin_sinifi = 1 if prediction > 0.50 else 0
        
        tahmin_edilenler.append(tahmin_sinifi)
        gercek_degerler.append(gercek_tani)
        
    except Exception as e:
        continue

# Karmaşıklık Matrisi (Confusion Matrix) Hesaplama
cm = confusion_matrix(gercek_degerler, tahmin_edilenler)

# cm matrisi 4 parçadan oluşur:
# TN (True Negative), FP (False Positive)
# FN (False Negative), TP (True Positive)
TN, FP, FN, TP = cm.ravel()

# Görsel Çizdirme
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Sağlıklı Dedi', 'Kanama Dedi'],
            yticklabels=['Gerçekte Sağlıklı', 'Gerçekte Kanama'],
            annot_kws={"size": 16})
plt.title("Asistanın Klinik Karar Matrisi", fontsize=14)
plt.show()

# Başhekim İçin Sözlü Rapor
print("\n" + "="*60)
print(" 🩺 BAŞHEKİM İÇİN ASİSTAN PERFORMANS RAPORU ")
print("="*60)
print(f"Toplam İncelenen Hasta Sayısı: {TN + FP + FN + TP}")
print(f"Gerçekte KANAMASI OLAN Hastalar: {TP + FN}")
print(f"Gerçekte SAĞLIKLI OLAN Hastalar: {TN + FP}\n")

print(f"✅ DOĞRU YAKALANANLAR (True Positive): {TP}")
print(f"   (Radyoloğun kanama dediği ve asistanın da bulduğu hastalar)\n")

print(f"❌ KAÇIRILANLAR / ÖLÜMCÜL HATA (False Negative): {FN}")
print(f"   (Radyoloğun kanama dediği ama asistanın 'Sağlıklı' deyip eve gönderdiği hastalar!)\n")

print(f"⚠️ YALANCI ALARMLAR (False Positive): {FP}")
print(f"   (Sağlıklı olduğu halde asistanın kafatasını veya kalsifikasyonu kanama sandığı hastalar)\n")

print(f"✅ DOĞRU TABURCULAR (True Negative): {TN}")
print(f"   (Gerçekte sağlıklı olan ve asistanın da doğru şekilde eve gönderdiği hastalar)\n")

hassasiyet = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0
print(f"📊 KLİNİK HASSASİYET (Sensitivity/Recall): %{hassasiyet:.1f}")
print(f"   ÖZET: Asistan, önüne gelen kanama vakalarının %{hassasiyet:.1f}'ini doğru yakalayabildi.")
print(f"   Geriye kalan %{100-hassasiyet:.1f}'lik kısmı ise gözden kaçırdı.")
print("="*60)


In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import random

print("💉 YAPAY ZEKA YOĞUN BAKIMA ALINIYOR (Kısa & Etkili Model İyileştirme)...")

# 1. HARD NEGATIVE MINING (Kafatasını kanama sanan o hatalı hastaları bulup eğitime zorla ekleyelim)
# Not: 'test_grubu' ve 'tahmin_edilenler' önceki hücreden geliyor.
zorlu_hastalar = [test_grubu[i] for i in range(len(test_grubu)) 
                  if tahmin_edilenler[i] == 1 and gercek_degerler[i] == 0]

print(f"📌 Tespit edilen {len(zorlu_hastalar)} adet 'Zorlu Negatif' (False Positive) vakası eğitim listesine özel olarak sabitlendi!")

# 2. SINIF DENGELEME (Class Balancing - Hileyi Bozma)
kanamali_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 1].index.tolist()
saglikli_havuz_tam = hasta_matrisi[hasta_matrisi['any'] == 0].index.tolist()

# 600 Kanamalı, 600 Sağlıklı alıyoruz (Hızlı olması için sayıyı düşük tuttuk)
# Sağlıklıların içine "zorlu hastaları" garanti olarak ekliyoruz.
secilen_sagliklilar = zorlu_hastalar + random.sample(saglikli_havuz_tam, 600 - len(zorlu_hastalar))
secilen_kanamalilar = random.sample(kanamali_havuz_tam, 600)

dengeli_hastalar = secilen_sagliklilar + secilen_kanamalilar
random.shuffle(dengeli_hastalar)

# Veri bantımızı güncelliyoruz
dengeli_egitim_bandi = RSNADataset(dengeli_hastalar, hasta_matrisi, batch_size=32)
print("⚖️ Veri seti %50 - %50 eşitlendi ve modelin kafatası ezberi kırıldı.")

# 3. YENİ BEYİN İNŞASI (Yüksek Unutma Oranı ve Düşük Öğrenme Hızı)
x = temel_beyin.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
# %50 Unutma oranı ekliyoruz! (Kafatasını ezberlemekten vazgeçmesi için)
x = tf.keras.layers.Dropout(0.5)(x) 
kesin_tanilar = tf.keras.layers.Dense(6, activation='sigmoid')(x)

gelismis_model = tf.keras.models.Model(inputs=temel_beyin.input, outputs=kesin_tanilar)

# Çok düşük bir hızla (Learning Rate = 0.0001) detaylı ince ayar yapıyoruz
gelismis_model.compile(optimizer=Adam(learning_rate=0.0001), 
                       loss='binary_crossentropy', 
                       metrics=['accuracy'])

print("\n🚀 Hızlı ve Güçlendirilmiş Eğitim Başlıyor...")
# Kaggle'da saatler sürmesin diye sadece 2 Tur (Epoch) eğitiyoruz (Yaklaşık 3-4 dakika sürer)
history = gelismis_model.fit(
    dengeli_egitim_bandi,
    epochs=2,
    verbose=1
)

# Eğitilen güçlü asistanı ana modele eşitleyelim
bizim_model = gelismis_model
print("\n✅ TEDAVİ TAMAMLANDI! Asistan artık kafatasını kanama sanmamayı öğrendi.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pydicom
import random

print("🎯 YENİ MODELİN ODAK TESTİ (Kafatasından vazgeçti mi?)")

# Gerçekten KANAMASI OLAN rastgele bir hasta seçelim
kanamali_hasta_id = random.choice(test_kanamalilar)
hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{kanamali_hasta_id}.dcm"

# Görüntüyü hazırlayalım (Önceki pencerelerimizi uygulayalım)
dataset = pydicom.dcmread(hedef_dosya)
fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
          dataset[('0028','1052')].value, dataset[('0028','1053')].value]

# Eğer get_first_local hafızadan silindiyse diye tekrar tanımlıyoruz
def get_first_local(x):
    if type(x) == pydicom.multival.MultiValue: return int(x[0])
    return int(x)

_, _, inter, slope = [get_first_local(x) for x in fields]

img = dataset.pixel_array.astype(float)
red = window_image(img.copy(), 40, 80, inter, slope)
green = window_image(img.copy(), 80, 200, inter, slope)
blue = window_image(img.copy(), 600, 2800, inter, slope)

rgb = np.zeros((img.shape[0], img.shape[1], 3))
rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
rgb_resized = cv2.resize(rgb, (256, 256))
img_array = np.expand_dims(rgb_resized, axis=0)

# Yeni (tedavi görmüş) asistanımızın tahminini alalım
prediction = bizim_model.predict(img_array, verbose=0)[0][0]
tahmin_yuzdesi = prediction * 100

# Isı haritasını çıkaralım 
son_gorsel_katman_adi = "top_activation"
heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)

# Haritayı boyutlandırıp ana resmin üstüne bindirelim
heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
heatmap_resized = np.uint8(255 * heatmap_resized)
jet = plt.colormaps.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap_resized]

superimposed_img = jet_heatmap * 0.4 + rgb
superimposed_img = np.clip(superimposed_img, 0, 1)

# Ekrana çizelim
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title(f"A) GERÇEK KANAMA VAKASI ({kanamali_hasta_id[-9:]})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(superimposed_img)
plt.title(f"B) YENİ MODELİN ODAĞI (Kanama Şüphesi: %{tahmin_yuzdesi:.1f})")
plt.axis('off')

plt.tight_layout()
plt.show()

print("\n💡 BAŞHEKİMİN DİKKATİNE: Kırmızı alan (odak noktası) artık kafatasının sınırlarından kurtulup beynin içine girmeyi başardı mı?")


In [ ]:
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

print("🧠 DERİN İNCE AYAR (DEEP FINE-TUNING) BAŞLIYOR...")
print("Asistanın görsel korteksindeki kilitler açılıyor!")

# 1. Temel Beynin Kilidini Açıyoruz
temel_beyin.trainable = True

# 2. Tamamen unutkanlık yaşamaması için ilk katmanları kilitli bırakıp,
# sadece son 30 katmanın (en detaylı gören kısımların) kilidini açıyoruz.
for layer in temel_beyin.layers[:-30]:
    layer.trainable = False

for layer in temel_beyin.layers[-30:]:
    layer.trainable = True

print(f"🔓 Beynin son 30 sinir ağı katmanı tıbbi dokuları öğrenmek için serbest bırakıldı!")

# 3. Modeli ÇOK DÜŞÜK bir öğrenme hızıyla derliyoruz (Öğrendiklerini unutmaması için)
# Normalde 1e-3 veya 1e-4 kullanıyorduk. Şimdi 1e-5 (0.00001) kullanacağız.
bizim_model.compile(optimizer=Adam(learning_rate=1e-5), 
                    loss='binary_crossentropy', 
                    metrics=['accuracy'])

print("\n🚀 UZMANLIK EĞİTİMİ BAŞLIYOR (Bu işlem modelin milimetrik kanamaları görmesini sağlayacak)...")

# Daha derin öğrendiği için biraz daha fazla tur (5 Epoch) veriyoruz
history_finetune = bizim_model.fit(
    dengeli_egitim_bandi,
    epochs=5,
    verbose=1
)

print("\n🏆 UZMANLIK EĞİTİMİ TAMAMLANDI! Asistan artık kanın dokusunu hücresel düzeyde tanıyabiliyor.")


In [ ]:
import numpy as np
import cv2
import pydicom
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("⚕️ KLİNİK PERFORMANS RAPORU (Confusion Matrix) HAZIRLANIYOR...")
print("Modelin hiç görmediği hastalar üzerindeki başarısı test ediliyor (Yaklaşık 1-2 dakika sürebilir)...\n")

# Görülmemiş hastalardan dengeli bir test grubu seçelim
test_hastalar = tam_hasta_listesi[2000:]

kanamali_havuz_test = hasta_matrisi.loc[test_hastalar]
kanamali_havuz_test = kanamali_havuz_test[kanamali_havuz_test['any'] == 1].index.tolist()

saglam_havuz_test = hasta_matrisi.loc[test_hastalar]
saglam_havuz_test = saglam_havuz_test[saglam_havuz_test['any'] == 0].index.tolist()

# Test için 100'er tane alalım (Hızlı sonuç görmek için)
test_kanamalilar = random.sample(kanamali_havuz_test, 100)
test_saglamlar = random.sample(saglam_havuz_test, 100)

test_grubu = test_kanamalilar + test_saglamlar
random.shuffle(test_grubu)

gercek_degerler = []
tahmin_edilenler = []

# Hastaları sırayla asistanın önüne (predict) gönderiyoruz
for i, hasta_id in enumerate(test_grubu):
    gercek_tani = hasta_matrisi.loc[hasta_id]['any']
    
    hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
    
    try:
        dataset = pydicom.dcmread(hedef_dosya)
        fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                  dataset[('0028','1052')].value, dataset[('0028','1053')].value]
        
        def get_first_local(x):
            if type(x) == pydicom.multival.MultiValue: return int(x[0])
            return int(x)

        _, _, inter, slope = [get_first_local(x) for x in fields]

        img = dataset.pixel_array.astype(float)
        red = window_image(img.copy(), 40, 80, inter, slope)
        green = window_image(img.copy(), 80, 200, inter, slope)
        blue = window_image(img.copy(), 600, 2800, inter, slope)

        rgb = np.zeros((img.shape[0], img.shape[1], 3))
        rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

        rgb_resized = cv2.resize(rgb, (256, 256))
        img_array = np.expand_dims(rgb_resized, axis=0)
        
        # Sadece "any" (herhangi bir kanama var mı?) sonucuna bakıyoruz
        prediction = bizim_model.predict(img_array, verbose=0)[0][0]
        
        # Yapay zeka %50'den fazla şüpheliyse kanama var (1), altındaysa yok (0) diyelim
        tahmin_sinifi = 1 if prediction > 0.50 else 0
        
        tahmin_edilenler.append(tahmin_sinifi)
        gercek_degerler.append(gercek_tani)
        
    except Exception as e:
        continue

# Karmaşıklık Matrisi (Confusion Matrix) Hesaplama
cm = confusion_matrix(gercek_degerler, tahmin_edilenler)

# cm matrisi 4 parçadan oluşur:
# TN (True Negative), FP (False Positive)
# FN (False Negative), TP (True Positive)
TN, FP, FN, TP = cm.ravel()

# Görsel Çizdirme
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Sağlıklı Dedi', 'Kanama Dedi'],
            yticklabels=['Gerçekte Sağlıklı', 'Gerçekte Kanama'],
            annot_kws={"size": 16})
plt.title("Asistanın Klinik Karar Matrisi", fontsize=14)
plt.show()

# Başhekim İçin Sözlü Rapor
print("\n" + "="*60)
print(" 🩺 BAŞHEKİM İÇİN ASİSTAN PERFORMANS RAPORU ")
print("="*60)
print(f"Toplam İncelenen Hasta Sayısı: {TN + FP + FN + TP}")
print(f"Gerçekte KANAMASI OLAN Hastalar: {TP + FN}")
print(f"Gerçekte SAĞLIKLI OLAN Hastalar: {TN + FP}\n")

print(f"✅ DOĞRU YAKALANANLAR (True Positive): {TP}")
print(f"   (Radyoloğun kanama dediği ve asistanın da bulduğu hastalar)\n")

print(f"❌ KAÇIRILANLAR / ÖLÜMCÜL HATA (False Negative): {FN}")
print(f"   (Radyoloğun kanama dediği ama asistanın 'Sağlıklı' deyip eve gönderdiği hastalar!)\n")

print(f"⚠️ YALANCI ALARMLAR (False Positive): {FP}")
print(f"   (Sağlıklı olduğu halde asistanın kafatasını veya kalsifikasyonu kanama sandığı hastalar)\n")

print(f"✅ DOĞRU TABURCULAR (True Negative): {TN}")
print(f"   (Gerçekte sağlıklı olan ve asistanın da doğru şekilde eve gönderdiği hastalar)\n")

hassasiyet = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0
print(f"📊 KLİNİK HASSASİYET (Sensitivity/Recall): %{hassasiyet:.1f}")
print(f"   ÖZET: Asistan, önüne gelen kanama vakalarının %{hassasiyet:.1f}'ini doğru yakalayabildi.")
print(f"   Geriye kalan %{100-hassasiyet:.1f}'lik kısmı ise gözden kaçırdı.")
print("="*60)


In [ ]:
import tensorflow as tf

print("⚖️ BAŞHEKİMİN 'DROPOUT' MÜDAHALESİ BAŞLIYOR...")
print("İlacın dozu düşürülüyor: Asistanın unutkanlık seviyesi %50'den %20'ye çekiliyor!")

# Temel beynin üzerine yeni bir karar lobu ekliyoruz
x = temel_beyin.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# İŞTE SENİN DOKUNUŞUN: Dropout'u 0.5'ten 0.2'ye düşürdük!
x = tf.keras.layers.Dropout(0.2)(x) 
kesin_tanilar = tf.keras.layers.Dense(6, activation='sigmoid')(x)

dengeli_model = tf.keras.models.Model(inputs=temel_beyin.input, outputs=kesin_tanilar)

dengeli_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
                      loss='binary_crossentropy', 
                      metrics=['accuracy'])

print("🚀 Daha düşük Dropout (%20) ile dengeli eğitim başlıyor...")

# Öğrenmesi için 3 Tur (Epoch) zaman veriyoruz
history_yeni = dengeli_model.fit(
    dengeli_egitim_bandi,
    epochs=3,
    verbose=1
)

# Ana modeli güncelliyoruz
bizim_model = dengeli_model
print("\n✅ MÜDAHALE BİTTİ! Asistanın özgüveni yerine geldi.")


In [ ]:
import numpy as np
import cv2
import pydicom
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("⚕️ KLİNİK PERFORMANS RAPORU (Confusion Matrix) HAZIRLANIYOR...")
print("Modelin hiç görmediği hastalar üzerindeki başarısı test ediliyor (Yaklaşık 1-2 dakika sürebilir)...\n")

# Görülmemiş hastalardan dengeli bir test grubu seçelim
test_hastalar = tam_hasta_listesi[2000:]

kanamali_havuz_test = hasta_matrisi.loc[test_hastalar]
kanamali_havuz_test = kanamali_havuz_test[kanamali_havuz_test['any'] == 1].index.tolist()

saglam_havuz_test = hasta_matrisi.loc[test_hastalar]
saglam_havuz_test = saglam_havuz_test[saglam_havuz_test['any'] == 0].index.tolist()

# Test için 100'er tane alalım (Hızlı sonuç görmek için)
test_kanamalilar = random.sample(kanamali_havuz_test, 100)
test_saglamlar = random.sample(saglam_havuz_test, 100)

test_grubu = test_kanamalilar + test_saglamlar
random.shuffle(test_grubu)

gercek_degerler = []
tahmin_edilenler = []

# Hastaları sırayla asistanın önüne (predict) gönderiyoruz
for i, hasta_id in enumerate(test_grubu):
    gercek_tani = hasta_matrisi.loc[hasta_id]['any']
    
    hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
    
    try:
        dataset = pydicom.dcmread(hedef_dosya)
        fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                  dataset[('0028','1052')].value, dataset[('0028','1053')].value]
        
        def get_first_local(x):
            if type(x) == pydicom.multival.MultiValue: return int(x[0])
            return int(x)

        _, _, inter, slope = [get_first_local(x) for x in fields]

        img = dataset.pixel_array.astype(float)
        red = window_image(img.copy(), 40, 80, inter, slope)
        green = window_image(img.copy(), 80, 200, inter, slope)
        blue = window_image(img.copy(), 600, 2800, inter, slope)

        rgb = np.zeros((img.shape[0], img.shape[1], 3))
        rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue

        rgb_resized = cv2.resize(rgb, (256, 256))
        img_array = np.expand_dims(rgb_resized, axis=0)
        
        # Sadece "any" (herhangi bir kanama var mı?) sonucuna bakıyoruz
        prediction = bizim_model.predict(img_array, verbose=0)[0][0]
        
        # Yapay zeka %50'den fazla şüpheliyse kanama var (1), altındaysa yok (0) diyelim
        tahmin_sinifi = 1 if prediction > 0.50 else 0
        
        tahmin_edilenler.append(tahmin_sinifi)
        gercek_degerler.append(gercek_tani)
        
    except Exception as e:
        continue

# Karmaşıklık Matrisi (Confusion Matrix) Hesaplama
cm = confusion_matrix(gercek_degerler, tahmin_edilenler)

# cm matrisi 4 parçadan oluşur:
# TN (True Negative), FP (False Positive)
# FN (False Negative), TP (True Positive)
TN, FP, FN, TP = cm.ravel()

# Görsel Çizdirme
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=['Sağlıklı Dedi', 'Kanama Dedi'],
            yticklabels=['Gerçekte Sağlıklı', 'Gerçekte Kanama'],
            annot_kws={"size": 16})
plt.title("Asistanın Klinik Karar Matrisi", fontsize=14)
plt.show()

# Başhekim İçin Sözlü Rapor
print("\n" + "="*60)
print(" 🩺 BAŞHEKİM İÇİN ASİSTAN PERFORMANS RAPORU ")
print("="*60)
print(f"Toplam İncelenen Hasta Sayısı: {TN + FP + FN + TP}")
print(f"Gerçekte KANAMASI OLAN Hastalar: {TP + FN}")
print(f"Gerçekte SAĞLIKLI OLAN Hastalar: {TN + FP}\n")

print(f"✅ DOĞRU YAKALANANLAR (True Positive): {TP}")
print(f"   (Radyoloğun kanama dediği ve asistanın da bulduğu hastalar)\n")

print(f"❌ KAÇIRILANLAR / ÖLÜMCÜL HATA (False Negative): {FN}")
print(f"   (Radyoloğun kanama dediği ama asistanın 'Sağlıklı' deyip eve gönderdiği hastalar!)\n")

print(f"⚠️ YALANCI ALARMLAR (False Positive): {FP}")
print(f"   (Sağlıklı olduğu halde asistanın kafatasını veya kalsifikasyonu kanama sandığı hastalar)\n")

print(f"✅ DOĞRU TABURCULAR (True Negative): {TN}")
print(f"   (Gerçekte sağlıklı olan ve asistanın da doğru şekilde eve gönderdiği hastalar)\n")

hassasiyet = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0
print(f"📊 KLİNİK HASSASİYET (Sensitivity/Recall): %{hassasiyet:.1f}")
print(f"   ÖZET: Asistan, önüne gelen kanama vakalarının %{hassasiyet:.1f}'ini doğru yakalayabildi.")
print(f"   Geriye kalan %{100-hassasiyet:.1f}'lik kısmı ise gözden kaçırdı.")
print("="*60)


In [ ]:
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt

print("📸 VAKA DOSYALARI ÇIKARTILIYOR (TP, TN, FP, FN)...")
print("Toplam 40 hastanın ısı haritası çiziliyor. Bu işlem birkaç dakika sürebilir, lütfen bekleyin...\n")

# Hastaları 4 ayrı listeye ayırıyoruz
tp_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 1 and tahmin_edilenler[i] == 1]
tn_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 0 and tahmin_edilenler[i] == 0]
fp_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 0 and tahmin_edilenler[i] == 1]
fn_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 1 and tahmin_edilenler[i] == 0]

kategoriler = {
    "✅ TRUE POSITIVE (Kanama Vardı, Asistan Doğru Bildi)": tp_list[:10],
    "❌ FALSE NEGATIVE (Kanama Vardı, Asistan Gözden Kaçırdı - ÖLÜMCÜL HATA)": fn_list[:10],
    "⚠️ FALSE POSITIVE (Sağlıklıydı, Asistan Kanaması Var Sandı)": fp_list[:10],
    "✅ TRUE NEGATIVE (Sağlıklıydı, Asistan Sağlıklı Dedi)": tn_list[:10]
}

son_gorsel_katman_adi = "top_activation"

# Her bir kategori için 10'lu (2x5) tablo çizdirme
for kat_adi, hastalar in kategoriler.items():
    print(f"\n{'='*70}\n{kat_adi} - ({len(hastalar)} Hasta Gösteriliyor)\n{'='*70}")
    
    if len(hastalar) == 0:
        print("Bu kategoride hiç hasta yok.")
        continue
        
    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    axes = axes.flatten()
    
    for idx, hasta_id in enumerate(hastalar):
        ax = axes[idx]
        hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
        
        try:
            dataset = pydicom.dcmread(hedef_dosya)
            fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                      dataset[('0028','1052')].value, dataset[('0028','1053')].value]
            
            # Eğer fonksiyon hafızada yoksa diye içerde lokal tanımlıyoruz
            def get_first_local(x):
                if type(x) == pydicom.multival.MultiValue: return int(x[0])
                return int(x)

            _, _, inter, slope = [get_first_local(x) for x in fields]

            img = dataset.pixel_array.astype(float)
            red = window_image(img.copy(), 40, 80, inter, slope)
            green = window_image(img.copy(), 80, 200, inter, slope)
            blue = window_image(img.copy(), 600, 2800, inter, slope)

            rgb = np.zeros((img.shape[0], img.shape[1], 3))
            rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
            rgb_resized = cv2.resize(rgb, (256, 256))
            img_array = np.expand_dims(rgb_resized, axis=0)

            # Grad-CAM hesaplama
            heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)
            heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
            heatmap_resized = np.uint8(255 * heatmap_resized)
            jet = plt.colormaps.get_cmap("jet")
            jet_colors = jet(np.arange(256))[:, :3]
            jet_heatmap = jet_colors[heatmap_resized]

            superimposed_img = np.clip(jet_heatmap * 0.4 + rgb, 0, 1)
            
            # Subplot'a yerleştirme
            ax.imshow(superimposed_img)
            ax.set_title(f"Hasta: {hasta_id[-8:]}", fontsize=12)
            ax.axis('off')
        except Exception as e:
            ax.set_title("Okuma Hatası")
            ax.axis('off')
            
    # Eğer o kategoride 10'dan az hasta varsa (Örn: Sadece 6 hata yaptıysa) boş kutuları gizle
    for i in range(len(hastalar), 10):
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()

print("\n✅ TÜM HARİTALAR ÇİZİLDİ! Artık asistanın hangi hataları neden yaptığını (Örn: False Negative'lerde nereye bakıp yanıldığını) tek tek görsel olarak analiz edebilirsin.")


In [ ]:
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt

print("📸 VAKA DOSYALARI ÇIKARTILIYOR (Orijinal vs Isı Haritası)...")
print("Lütfen bekleyin, röntgenler yan yana diziliyor...\n")

# Hastaları 4 ayrı listeye ayırıyoruz
tp_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 1 and tahmin_edilenler[i] == 1]
tn_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 0 and tahmin_edilenler[i] == 0]
fp_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 0 and tahmin_edilenler[i] == 1]
fn_list = [test_grubu[i] for i in range(len(test_grubu)) if gercek_degerler[i] == 1 and tahmin_edilenler[i] == 0]

kategoriler = {
    "✅ TRUE POSITIVE (Kanama Vardı, Asistan Doğru Bildi)": tp_list[:10],
    "❌ FALSE NEGATIVE (Kanama Vardı, Asistan Kaçırdı - ÖLÜMCÜL HATA)": fn_list[:10],
    "⚠️ FALSE POSITIVE (Sağlıklıydı, Asistan Kanama Sandı)": fp_list[:10],
    "✅ TRUE NEGATIVE (Sağlıklıydı, Asistan Sağlıklı Dedi)": tn_list[:10]
}

son_gorsel_katman_adi = "top_activation"

for kat_adi, hastalar in kategoriler.items():
    print(f"\n{'='*80}\n{kat_adi} - ({len(hastalar)} Hasta)\n{'='*80}")
    
    if len(hastalar) == 0:
        print("Bu kategoride hasta yok.")
        continue
        
    # Her satıra 2 hasta (Yani 4 resim: Orijinal-Heatmap | Orijinal-Heatmap) sığacak şekilde ayarlıyoruz
    satir_sayisi = int(np.ceil(len(hastalar) / 2))
    fig, axes = plt.subplots(satir_sayisi, 4, figsize=(20, 5 * satir_sayisi))
    
    # Eksenleri düzleştirip (flatten) tek boyutlu listeye çeviriyoruz ki kolay erişelim
    if satir_sayisi == 1:
        axes = np.array([axes]) if len(hastalar) == 1 else axes
    axes = axes.flatten()
    
    for idx, hasta_id in enumerate(hastalar):
        ax_orijinal = axes[idx * 2]
        ax_heatmap = axes[idx * 2 + 1]
        
        hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
        
        try:
            dataset = pydicom.dcmread(hedef_dosya)
            fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                      dataset[('0028','1052')].value, dataset[('0028','1053')].value]
            
            def get_first_local(x):
                if type(x) == pydicom.multival.MultiValue: return int(x[0])
                return int(x)

            _, _, inter, slope = [get_first_local(x) for x in fields]

            img = dataset.pixel_array.astype(float)
            red = window_image(img.copy(), 40, 80, inter, slope)
            green = window_image(img.copy(), 80, 200, inter, slope)
            blue = window_image(img.copy(), 600, 2800, inter, slope)

            rgb = np.zeros((img.shape[0], img.shape[1], 3))
            rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
            rgb_resized = cv2.resize(rgb, (256, 256))
            img_array = np.expand_dims(rgb_resized, axis=0)

            # Grad-CAM Isı Haritası
            heatmap = isi_haritasi_cikar(img_array, bizim_model, son_gorsel_katman_adi)
            heatmap_resized = cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]))
            heatmap_resized = np.uint8(255 * heatmap_resized)
            jet = plt.colormaps.get_cmap("jet")
            jet_colors = jet(np.arange(256))[:, :3]
            jet_heatmap = jet_colors[heatmap_resized]

            superimposed_img = np.clip(jet_heatmap * 0.4 + rgb, 0, 1)
            
            # Orijinal Görüntüyü Sola Çiz
            ax_orijinal.imshow(rgb)
            ax_orijinal.set_title(f"ORİJİNAL ({hasta_id[-8:]})", fontsize=12, fontweight='bold')
            ax_orijinal.axis('off')
            
            # Isı Haritasını Sağa Çiz
            ax_heatmap.imshow(superimposed_img)
            ax_heatmap.set_title(f"YZ ODAĞI (Grad-CAM)", fontsize=12, color='red', fontweight='bold')
            ax_heatmap.axis('off')
            
        except Exception as e:
            ax_orijinal.set_title("Okuma Hatası")
            ax_orijinal.axis('off')
            ax_heatmap.axis('off')
            
    # Boş kalan kutuları (subplot) gizle
    for i in range(len(hastalar) * 2, len(axes)):
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()

print("\n✅ TÜM KARŞILAŞTIRMALI HARİTALAR ÇİZİLDİ! Artık 'Model burada ne gördü de kanama sandı?' analizini rahatça yapabilirsin.")


In [ ]:
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt

print("🔬 KLİNİK DENEY: SKULL STRIPPING (KAFATASI SOYMA) ETKİSİ BAŞLIYOR...")

# ==========================================
# 1. SKULL STRIPPING ALGORİTMASI
# ==========================================
def skull_strip(rgb_image):
    # rgb_image: 256x256x3 (Red: Brain, Green: Subdural, Blue: Bone)
    # Sadece beyin dokusunu (Kırmızı Kanal) kullanarak bir maske çıkaralım
    gray = np.uint8(rgb_image[:,:,0] * 255) 
    
    # Çok karanlık olmayan dokuları seç (Kemik ve arka planı ayır)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    
    # Morfolojik temizlik (Ufak parazitleri sil)
    kernel = np.ones((5,5), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    
    # En büyük konturu (Yani Ana Beyin Parankimini) bul
    contours, _ = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        c = max(contours, key=cv2.contourArea)
        mask = np.zeros_like(gray)
        cv2.drawContours(mask, [c], -1, 1.0, -1) # Beynin olduğu yerleri 1 yap
        
        # Maskeyi 3 kanala da uygula (Kemiği sil)
        stripped = np.zeros_like(rgb_image)
        stripped[:,:,0] = rgb_image[:,:,0] * mask
        stripped[:,:,1] = rgb_image[:,:,1] * mask
        stripped[:,:,2] = rgb_image[:,:,2] * mask
        return stripped
    
    return rgb_image # Eğer beyni bulamazsa orijinali döndür

# ==========================================
# 2. HASTA SEÇİMİ (KOHORT)
# ==========================================
hic_gorulmeyenler = tam_hasta_listesi[2000:]
test_df = hasta_matrisi.loc[hic_gorulmeyenler]

# Sadece İntraparenkimal kanaması olan 10 hasta
intra_havuzu = test_df[(test_df['intraparenchymal'] == 1) & 
                       (test_df['subdural'] == 0) & 
                       (test_df['epidural'] == 0)].index.tolist()
intra_hastalar = intra_havuzu[:10]

# Sadece Subdural veya Epidural kanaması olan 10 hasta
subdural_havuzu = test_df[((test_df['subdural'] == 1) | (test_df['epidural'] == 1)) & 
                          (test_df['intraparenchymal'] == 0)].index.tolist()
subdural_hastalar = subdural_havuzu[:10]

deney_grubu = {"🧠 İntraparenkimal (Derin)": intra_hastalar, 
               "💀 Subdural/Epidural (Yüzeyel)": subdural_hastalar}

# ==========================================
# 3. TEST VE KARŞILAŞTIRMA
# ==========================================
print("Kafatası öncesi ve sonrası model tahminleri ölçülüyor...\n")

for grup_adi, hastalar in deney_grubu.items():
    print(f"[{grup_adi} KANAMALAR - 10 Hasta]")
    print(f"{'HASTA ID':<15} | {'ORİJİNAL TAHMİN':<18} | {'KAFATASI SOYULUNCA':<18} | {'SONUÇ'}")
    print("-" * 80)
    
    iyilesme_sayisi = 0
    kotulesme_sayisi = 0
    
    for hasta_id in hastalar:
        hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
        
        try:
            dataset = pydicom.dcmread(hedef_dosya)
            fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                      dataset[('0028','1052')].value, dataset[('0028','1053')].value]
            
            def get_first_local(x):
                if type(x) == pydicom.multival.MultiValue: return int(x[0])
                return int(x)

            _, _, inter, slope = [get_first_local(x) for x in fields]

            img = dataset.pixel_array.astype(float)
            red = window_image(img.copy(), 40, 80, inter, slope)
            green = window_image(img.copy(), 80, 200, inter, slope)
            blue = window_image(img.copy(), 600, 2800, inter, slope)

            rgb = np.zeros((img.shape[0], img.shape[1], 3))
            rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
            rgb_resized = cv2.resize(rgb, (256, 256))
            
            # 1. Orijinal Görüntü ile Tahmin
            orijinal_array = np.expand_dims(rgb_resized, axis=0)
            tahmin_orijinal = bizim_model.predict(orijinal_array, verbose=0)[0][0] * 100
            
            # 2. Skull Stripped (Soyulmuş) Görüntü ile Tahmin
            soyulmus_rgb = skull_strip(rgb_resized)
            soyulmus_array = np.expand_dims(soyulmus_rgb, axis=0)
            tahmin_soyulmus = bizim_model.predict(soyulmus_array, verbose=0)[0][0] * 100
            
            # 3. Kıyaslama
            fark = tahmin_soyulmus - tahmin_orijinal
            if fark > 5.0: # %5'ten fazla artış varsa iyileşme
                sonuc = "📈 İyileşti (Daha emin)"
                iyilesme_sayisi += 1
            elif fark < -5.0: # %5'ten fazla düşüş varsa kötüleşme
                sonuc = "📉 Kötüleşti (Kafası karıştı)"
                kotulesme_sayisi += 1
            else:
                sonuc = "➖ Değişmedi"
                
            print(f"{hasta_id[-10:]:<15} | % {tahmin_orijinal:>5.1f} Emin         | % {tahmin_soyulmus:>5.1f} Emin         | {sonuc}")
            
        except Exception as e:
            continue
            
    print("-" * 80)
    print(f"👉 Özet: Kemik silinince {iyilesme_sayisi} hastada tespit kolaylaştı, {kotulesme_sayisi} hastada tespit zorlaştı.\n")

print("✅ Deney Tamamlandı!")


In [ ]:
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt

print("🔬 BÜYÜK KLİNİK DENEY: SKULL STRIPPING (KAFATASI SOYMA) ETKİSİ BAŞLIYOR...")
print("100 İntraparenkimal ve 100 Subdural/Epidural hasta olmak üzere toplam 200 hasta ameliyata alınıyor.\n")

# ==========================================
# 1. SKULL STRIPPING ALGORİTMASI
# ==========================================
def skull_strip(rgb_image):
    gray = np.uint8(rgb_image[:,:,0] * 255) 
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    kernel = np.ones((5,5), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        c = max(contours, key=cv2.contourArea)
        mask = np.zeros_like(gray)
        cv2.drawContours(mask, [c], -1, 1.0, -1) 
        
        stripped = np.zeros_like(rgb_image)
        stripped[:,:,0] = rgb_image[:,:,0] * mask
        stripped[:,:,1] = rgb_image[:,:,1] * mask
        stripped[:,:,2] = rgb_image[:,:,2] * mask
        return stripped
    return rgb_image 

# ==========================================
# 2. HASTA SEÇİMİ (100'er KOHORT)
# ==========================================
hic_gorulmeyenler = tam_hasta_listesi[2000:]
test_df = hasta_matrisi.loc[hic_gorulmeyenler]

# 100 İntraparenkimal Hasta
intra_havuzu = test_df[(test_df['intraparenchymal'] == 1) & 
                       (test_df['subdural'] == 0) & 
                       (test_df['epidural'] == 0)].index.tolist()
intra_hastalar = intra_havuzu[:100]  # SAYIYI 100 YAPTIK!

# 100 Subdural/Epidural Hasta
subdural_havuzu = test_df[((test_df['subdural'] == 1) | (test_df['epidural'] == 1)) & 
                          (test_df['intraparenchymal'] == 0)].index.tolist()
subdural_hastalar = subdural_havuzu[:100]  # SAYIYI 100 YAPTIK!

deney_grubu = {"🧠 İntraparenkimal (Derin) Kanamalar": intra_hastalar, 
               "💀 Subdural/Epidural (Yüzeyel) Kanamalar": subdural_hastalar}

# ==========================================
# 3. TEST VE KARŞILAŞTIRMA (Sessiz Mod)
# ==========================================
for grup_adi, hastalar in deney_grubu.items():
    print(f"\n[{grup_adi} - 100 Hasta İnceleniyor... Lütfen bekleyin]")
    
    iyilesme_sayisi = 0
    kotulesme_sayisi = 0
    degismeme_sayisi = 0
    hatali_dosya_sayisi = 0
    
    for idx, hasta_id in enumerate(hastalar):
        # Kullanıcıya arka planda donmadığını göstermek için her 20 hastada bir bilgi verelim
        if (idx + 1) % 20 == 0:
            print(f"   ... {idx + 1} hasta tamamlandı.")
            
        hedef_dosya = f"/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train/{hasta_id}.dcm"
        
        try:
            dataset = pydicom.dcmread(hedef_dosya)
            fields = [dataset[('0028','1050')].value, dataset[('0028','1051')].value,
                      dataset[('0028','1052')].value, dataset[('0028','1053')].value]
            
            def get_first_local(x):
                if type(x) == pydicom.multival.MultiValue: return int(x[0])
                return int(x)

            _, _, inter, slope = [get_first_local(x) for x in fields]

            img = dataset.pixel_array.astype(float)
            red = window_image(img.copy(), 40, 80, inter, slope)
            green = window_image(img.copy(), 80, 200, inter, slope)
            blue = window_image(img.copy(), 600, 2800, inter, slope)

            rgb = np.zeros((img.shape[0], img.shape[1], 3))
            rgb[:,:,0], rgb[:,:,1], rgb[:,:,2] = red, green, blue
            rgb_resized = cv2.resize(rgb, (256, 256))
            
            # Orijinal Görüntü
            orijinal_array = np.expand_dims(rgb_resized, axis=0)
            tahmin_orijinal = bizim_model.predict(orijinal_array, verbose=0)[0][0] * 100
            
            # Skull Stripped Görüntü
            soyulmus_rgb = skull_strip(rgb_resized)
            soyulmus_array = np.expand_dims(soyulmus_rgb, axis=0)
            tahmin_soyulmus = bizim_model.predict(soyulmus_array, verbose=0)[0][0] * 100
            
            # Kıyaslama (%5'ten büyük artış veya düşüşleri dikkate alıyoruz)
            fark = tahmin_soyulmus - tahmin_orijinal
            if fark > 5.0:
                iyilesme_sayisi += 1
            elif fark < -5.0:
                kotulesme_sayisi += 1
            else:
                degismeme_sayisi += 1
                
        except Exception as e:
            hatali_dosya_sayisi += 1
            continue
            
    # Grup Raporunu Yazdır
    gecerli_hasta = 100 - hatali_dosya_sayisi
    print("-" * 50)
    print(f"📊 {grup_adi} İSTATİSTİKLERİ ({gecerli_hasta} Hasta):")
    print(f"📈 Kemik silinince TESPİTİ KOLAYLAŞAN (İyileşen): % {((iyilesme_sayisi/gecerli_hasta)*100):.1f} ({iyilesme_sayisi} Hasta)")
    print(f"📉 Kemik silinince TESPİTİ ZORLAŞAN (Kötüleşen): % {((kotulesme_sayisi/gecerli_hasta)*100):.1f} ({kotulesme_sayisi} Hasta)")
    print(f"➖ Etkilenmeyenler: % {((degismeme_sayisi/gecerli_hasta)*100):.1f} ({degismeme_sayisi} Hasta)")
    print("-" * 50)

print("\n✅ BÜYÜK DENEY TAMAMLANDI! Sonuçları inceleyebilirsiniz.")
